# LavaLabs Atmospheric Lab — source-only reproduction
Code and aggregate results for OBSCURA. No competition rows, identifiers, chips, feature matrices or prediction outputs are embedded.
Measured saved private run: 372947210. Public score: 1.15808; private performance unknown.
To reproduce, make a PRIVATE copy, attach the official OBSCURA input and enable the guarded run below. Keep its CSV outputs private.
OpenAI Codex assisted implementation/tests/writing. Official 54-feature recipe and HistGB settings adapted from Zelzy's Apache-2.0 starter: https://www.kaggle.com/code/zelza0/obscura-starter-baseline .
Custom cloud/spatial variants were tested and rejected. Development family selection and confirmation acceptance both have selection optimism.


In [ ]:
from pathlib import Path
import json, os, sys
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
package = Path('/kaggle/working/atmospheric_lab')
package.mkdir(exist_ok=True)
sources = json.loads('{"__init__.py": "\\"\\"\\"LavaLabs Atmospheric Lab. SI-assisted code; scientific inputs retain provenance.\\"\\"\\"\\n", "baseline.py": "\\"\\"\\"First measured baseline. Official HistGB recipe, supplied features, no selection.\\"\\"\\"\\nfrom pathlib import Path\\nimport importlib.metadata\\nimport platform\\nimport time\\nimport numpy as np\\nfrom sklearn.ensemble import HistGradientBoostingRegressor\\nfrom threadpoolctl import threadpool_limits\\nfrom .core import TARGETS, SOURCE, load_tables, site_folds, metric, export_submission, write_json\\n\\nPARAMS = dict(max_iter=250, learning_rate=.04, max_depth=3,\\n              min_samples_leaf=40, l2_regularization=5., max_features=.6,\\n              random_state=0, early_stopping=False)\\n\\n\\ndef fit_predict(x, y, z, params=None):\\n    y = np.asarray(y, dtype=float)\\n    keep = np.isfinite(y)\\n    if keep.sum() < 2:\\n        raise ValueError(\'Not enough measured labels in the fitting partition\')\\n    with threadpool_limits(limits=2):\\n        model = HistGradientBoostingRegressor(**(PARAMS if params is None else params))\\n        model.fit(np.asarray(x)[keep], y[keep])\\n        prediction = model.predict(z)\\n    if not np.isfinite(prediction).all():\\n        raise ValueError(\'Estimator produced a nonfinite prediction\')\\n    return model, prediction\\n\\n\\ndef run(root, output, n_splits=5, params=None, expected_rows=1221):\\n    started = time.monotonic()\\n    output = Path(output)\\n    output.mkdir(parents=True, exist_ok=True)\\n    train, test, sample, names = load_tables(root, expected_rows)\\n    x = train[names].to_numpy(dtype=float)\\n    z = test[names].to_numpy(dtype=float)\\n    y = train[TARGETS].to_numpy(dtype=float)\\n    folds = site_folds(train.site_key.to_numpy(), n_splits)\\n    oof = np.full(y.shape, np.nan)\\n    mean_oof = np.full(y.shape, np.nan)\\n    fold_reports = []\\n    for fold_number, (fit, check) in enumerate(folds, 1):\\n        tick = time.monotonic()\\n        for j in range(5):\\n            _, oof[check, j] = fit_predict(x[fit], y[fit, j], x[check], params)\\n            mean_oof[check, j] = np.nanmean(y[fit, j])\\n        report = metric(y[check], oof[check])\\n        report.update(fold=fold_number, seconds=round(time.monotonic()-tick, 3),\\n                      heldout_sites=int(train.iloc[check].site_key.nunique()),\\n                      fit_sites=int(train.iloc[fit].site_key.nunique()))\\n        fold_reports.append(report)\\n        print(f\'Fold {fold_number}/{n_splits}: MCRMSE {report[\\"mcrmse\\"]:.6f}; {report[\\"seconds\\"]:.1f}s\', flush=True)\\n    predictions = np.zeros((len(test), 5))\\n    for j in range(5):\\n        _, predictions[:, j] = fit_predict(x, y[:, j], z, params)\\n    export_hash = export_submission(sample, predictions, output/\'submission_table.csv\')\\n    receipt = {\'schema\': 1, \'experiment\': \'supplied_table_unselected_histgb\',\\n               \'si_disclosure\': \'OpenAI Codex assisted code, tests and documentation; source observations retain competition provenance.\',\\n               \'baseline_reference\': SOURCE, \'baseline_adaptation\': \'29 supplied features; explicit no early stopping; no OOF-tuned shrinkage.\',\\n               \'counts\': {\'train\': len(train), \'test\': len(test), \'sites\': int(train.site_key.nunique()), \'features\': len(names)},\\n               \'validation\': {\'method\': \'whole-site GroupKFold\', \'folds\': n_splits, \'site_overlap\': False,\\n                              \'selection\': \'none; fixed baseline parameters\', \'pooled_model\': metric(y, oof),\\n                              \'pooled_fold_mean\': metric(y, mean_oof), \'pooled_zero\': metric(y, np.zeros_like(y)),\\n                              \'fold_reports\': fold_reports},\\n               \'parameters\': PARAMS if params is None else params, \'seconds\': round(time.monotonic()-started, 3),\\n               \'environment\': {\'python\': platform.python_version(), **{p: importlib.metadata.version(p) for p in [\'numpy\',\'pandas\',\'scikit-learn\',\'scipy\']}},\\n               \'submission\': {\'rows\': len(test), \'sha256\': export_hash, \'finite\': True, \'private_file\': \'submission_table.csv\'},\\n               \'boundary\': \'Unseen-site validation within training time; private/future-time performance is unknown. No leaderboard score yet.\'}\\n    write_json(output/\'baseline_receipt.json\', receipt)\\n    print(\'BASELINE_COMPLETE\', flush=True)\\n    print(receipt[\'validation\'][\'pooled_model\'], flush=True)\\n    return receipt\\n", "core.py": "\\"\\"\\"Contracts and numerical checks for masked, whole-site atmospheric regression.\\"\\"\\"\\nfrom pathlib import Path\\nimport hashlib\\nimport json\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.model_selection import GroupKFold\\n\\nTARGETS = [\'t_ch4\', \'t_no2\', \'t_co\', \'t_aai\', \'t_firms\']\\nSOURCE = \'https://www.kaggle.com/code/zelza0/obscura-starter-baseline\'\\n\\n\\ndef metric(truth, prediction):\\n    \\"\\"\\"Official pooled, masked target RMSEs; never ignore a bad prediction.\\"\\"\\"\\n    y, p = np.asarray(truth, dtype=float), np.asarray(prediction, dtype=float)\\n    if y.ndim != 2 or y.shape != p.shape or y.shape[1] != 5:\\n        raise ValueError(\'Expected equally shaped n-by-five targets and predictions\')\\n    if np.isinf(y).any() or not np.isfinite(p).all():\\n        raise ValueError(\'Targets cannot be infinite; every prediction must be finite\')\\n    present = np.isfinite(y)\\n    counts = present.sum(axis=0)\\n    if (counts == 0).any():\\n        raise ValueError(\'All five targets need at least one observed value to report MCRMSE\')\\n    errors = [float(np.sqrt(np.mean((y[present[:, j], j] - p[present[:, j], j]) ** 2)))\\n              for j in range(5)]\\n    return {\'mcrmse\': float(np.mean(errors)), \'rmse\': dict(zip(TARGETS, errors)),\\n            \'observed\': dict(zip(TARGETS, map(int, counts)))}\\n\\n\\ndef validate_tables(train, test, sample, expected_rows=None):\\n    for name, frame in [(\'train\', train), (\'test\', test), (\'sample\', sample)]:\\n        if frame.columns.duplicated().any() or \'sample_id\' not in frame:\\n            raise ValueError(f\'{name}: duplicate columns or missing identifier\')\\n        ids = frame[\'sample_id\']\\n        if ids.isna().any() or not ids.is_unique or not ids.map(lambda x: isinstance(x, str) and bool(x.strip())).all():\\n            raise ValueError(f\'{name}: identifiers must be unique, nonempty strings\')\\n    if set(train.sample_id) & set(test.sample_id):\\n        raise ValueError(\'Training and test sample IDs overlap\')\\n    if list(sample.columns) != [\'sample_id\', *TARGETS]:\\n        raise ValueError(\'Submission template columns do not match the contract\')\\n    if list(test.sample_id) != list(sample.sample_id):\\n        raise ValueError(\'Test and template identifier order differs\')\\n    if expected_rows is not None and len(test) != expected_rows:\\n        raise ValueError(\'Unexpected test row count\')\\n    if \'site_key\' not in train or train.site_key.isna().any():\\n        raise ValueError(\'Training sites must be present\')\\n    features = [c for c in test if c != \'sample_id\']\\n    if any(c in features for c in [\'site_key\', *TARGETS]):\\n        raise ValueError(\'Label or site identifier found among predictors\')\\n    if set(train.columns) != {\'sample_id\', \'site_key\', *TARGETS, *features}:\\n        raise ValueError(\'Training and test feature schemas differ\')\\n    for frame, names in [(train, [*features, *TARGETS]), (test, features)]:\\n        for name in names:\\n            if not pd.api.types.is_numeric_dtype(frame[name]) or np.isinf(frame[name].to_numpy(dtype=float)).any():\\n                raise ValueError(\'Predictors and targets must be numerical, missing or finite\')\\n    if train[TARGETS].notna().sum().eq(0).any():\\n        raise ValueError(\'A training target has no observations\')\\n    return features\\n\\n\\ndef load_tables(root, expected_rows=1221):\\n    root = Path(root)\\n    train = pd.read_csv(root / \'train.csv\', dtype={\'sample_id\': str, \'site_key\': str})\\n    test = pd.read_csv(root / \'test.csv\', dtype={\'sample_id\': str})\\n    sample = pd.read_csv(root / \'sample_submission.csv\', dtype={\'sample_id\': str})\\n    features = validate_tables(train, test, sample, expected_rows)\\n    if len(features) != 29:\\n        raise ValueError(\'Expected the official 29 supplied numerical features\')\\n    return train, test, sample, features\\n\\n\\ndef site_folds(groups, n_splits=5, seed=None):\\n    groups = np.asarray(groups)\\n    if pd.isna(groups).any() or len(set(groups)) < n_splits:\\n        raise ValueError(\'Not enough valid sites for the requested folds\')\\n    splitter = GroupKFold(n_splits=n_splits) if seed is None else GroupKFold(n_splits=n_splits,shuffle=True,random_state=seed)\\n    folds = list(splitter.split(np.zeros(len(groups)), groups=groups))\\n    seen = np.zeros(len(groups), dtype=int)\\n    for fit, check in folds:\\n        if set(groups[fit]) & set(groups[check]):\\n            raise AssertionError(\'Site leakage across a fold\')\\n        seen[check] += 1\\n    if not np.all(seen == 1):\\n        raise AssertionError(\'Every row must be held out exactly once\')\\n    return folds\\n\\n\\ndef export_submission(sample, predictions, destination):\\n    p = np.asarray(predictions, dtype=float)\\n    if p.shape != (len(sample), 5) or not np.isfinite(p).all():\\n        raise ValueError(\'Test predictions must be finite and match all template rows\')\\n    if not sample.sample_id.is_unique or not sample.sample_id.map(lambda x: isinstance(x, str)).all():\\n        raise ValueError(\'Invalid submission identifiers\')\\n    result = sample[[\'sample_id\']].copy()\\n    result[TARGETS] = p\\n    path = Path(destination)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    result.to_csv(path, index=False, float_format=\'%.9g\')\\n    replay = pd.read_csv(path, dtype={\'sample_id\': str})\\n    if list(replay.sample_id) != list(sample.sample_id) or not np.isfinite(replay[TARGETS]).all().all():\\n        raise AssertionError(\'Export round trip failed\')\\n    return hashlib.sha256(path.read_bytes()).hexdigest()\\n\\n\\ndef write_json(path, value):\\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\\\n\', encoding=\'utf-8\')\\n", "experiment.py": "\\"\\"\\"Bounded content-feature experiments and site-held-out confirmation on Kaggle.\\"\\"\\"\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom pathlib import Path\\nimport hashlib\\nimport importlib.metadata\\nimport inspect\\nimport json\\nimport platform\\nimport time\\nimport numpy as np\\nfrom sklearn.model_selection import GroupShuffleSplit\\nfrom . import features as feature_module\\nfrom .features import chip_summary, BAND_ORDER, _spatial\\nfrom .baseline import PARAMS\\nfrom .core import TARGETS, load_tables, site_folds, metric, write_json, export_submission\\nfrom .selection import nested_compare, final_predict\\n\\n\\ndef extract_one(path):\\n    import rasterio\\n    with rasterio.open(path) as source:\\n        chip=source.read()  # Intentionally do not inspect geospatial/file metadata.\\n    official=chip_summary(chip,\'official\')\\n    clear=(chip[3]>0)&~np.isin(chip[12],[3,8,9,10])\\n    sufficient=clear.sum()>=500\\n    if sufficient:\\n        revised=dict(official)\\n        revised.update(q_insufficient_clear=0.,q_clear_count=float(clear.sum()),q_cloud_high_fraction=float((chip[13]>=40).mean()))\\n    else:\\n        revised=chip_summary(chip,\'no_fallback\')\\n    bands={name:chip[index].astype(float)*1e-4 for index,name in enumerate(BAND_ORDER[:12])}\\n    spatial={**revised,**_spatial(bands,clear,bool(sufficient))}\\n    return official,revised,spatial\\n\\n\\ndef feature_matrices(root,ids,cache_path,workers=2):\\n    root,cache_path=Path(root),Path(cache_path)\\n    # Both functions define extraction; changing either must invalidate the cache.\\n    source_hash=hashlib.sha256((inspect.getsource(feature_module)+\'\\\\n\'+inspect.getsource(extract_one)).encode()).hexdigest()\\n    fingerprint=hashlib.sha256((source_hash+\'\\\\n\'+str(root.resolve())+\'\\\\n\'+\'\\\\n\'.join(ids)).encode()).hexdigest()\\n    if cache_path.exists():\\n        with np.load(cache_path,allow_pickle=False) as saved:\\n            if saved[\'fingerprint\'].item()==fingerprint:\\n                summary=json.loads(saved[\'summary\'].item())\\n                expected={\'official\':54,\'no_fallback\':57,\'spatial\':73}\\n                for name,width in expected.items():\\n                    value=saved[name]\\n                    names=summary[\'column_names\'][name]\\n                    if (value.shape!=(len(ids),width) or np.isinf(value).any()\\n                            or len(names)!=width or names!=sorted(set(names))\\n                            or summary[\'columns\'][name]!=width or summary[\'chip_count\']!=len(ids)\\n                            or summary[\'feature_source_sha256\']!=source_hash):\\n                        raise ValueError(\'Private feature cache failed schema/integrity checks; rebuild it.\')\\n                print(\'Verified private feature cache loaded.\',flush=True)\\n                return {name:saved[name].copy() for name in expected},summary\\n    rows={name:[] for name in [\'official\',\'no_fallback\',\'spatial\']}\\n    names={}\\n    started=time.monotonic()\\n    def worker(sid):\\n        try: return extract_one(root/\'images\'/f\'{sid}.tif\')\\n        except Exception:\\n            # Keep row identifiers and original path errors out of notebook outputs.\\n            raise RuntimeError(\'A competition image failed extraction; no fabricated fallback is permitted.\') from None\\n    with ThreadPoolExecutor(max_workers=workers) as pool:\\n        for number,triplet in enumerate(pool.map(worker,ids),1):\\n            for name,features in zip(rows,triplet):\\n                keys=sorted(features)\\n                if name not in names: names[name]=keys\\n                if keys!=names[name]: raise AssertionError(\'Feature schema changed across chips\')\\n                rows[name].append([features[k] for k in keys])\\n            if number%250==0 or number==len(ids):\\n                elapsed=time.monotonic()-started\\n                print(f\'Chips {number}/{len(ids)}; {elapsed:.1f}s; estimated {(len(ids)-number)*elapsed/max(number,1)/60:.1f} min remaining\',flush=True)\\n    matrices={name:np.asarray(value,dtype=np.float64) for name,value in rows.items()}\\n    for value in matrices.values():\\n        if np.isinf(value).any(): raise ValueError(\'Infinite extracted predictor\')\\n    revised=matrices[\'no_fallback\']\\n    summary={\'chip_count\':len(ids),\'columns\':{name:len(keys) for name,keys in names.items()},\\n             \'column_names\':names,\'insufficient_clear_fraction\':float(revised[:,names[\'no_fallback\'].index(\'q_insufficient_clear\')].mean()),\\n             \'seconds\':round(time.monotonic()-started,3),\'feature_source_sha256\':source_hash,\\n             \'boundary\':\'Relative pixels only; no geographic location, ground scale or outside observations.\'}\\n    cache_path.parent.mkdir(parents=True,exist_ok=True)\\n    np.savez_compressed(cache_path,**matrices,fingerprint=np.array(fingerprint),summary=np.array(json.dumps(summary)))\\n    return matrices,summary\\n\\n\\ndef _public_summary(comparison):\\n    return {key:comparison[key] for key in [\'scores\',\'folds\',\'fold_mean_score\',\'raw_scores\']}\\n\\n\\nGATE_TOLERANCE = 1e-10\\n\\n\\ndef selection_plan(scores):\\n    \\"\\"\\"Freeze one comparator and at most one novel challenger on development only.\\"\\"\\"\\n    values={name:float(value[\'mcrmse\']) for name,value in scores.items()}\\n    if set(values)!={\'table\',\'official\',\'no_fallback\',\'spatial\'} or not all(np.isfinite(list(values.values()))):\\n        raise ValueError(\'Need four finite development scores\')\\n    control=\'official\' if values[\'official\']<values[\'table\']-GATE_TOLERANCE else \'table\'\\n    novel=\'spatial\' if values[\'spatial\']<values[\'no_fallback\']-GATE_TOLERANCE else \'no_fallback\'\\n    eligible=values[novel]<values[control]-GATE_TOLERANCE\\n    return {\'control\':control,\'novel\':novel,\'challenger_eligible\':bool(eligible),\\n            \'candidate\':novel if eligible else control,\'tolerance\':GATE_TOLERANCE,\\n            \'gate_protocol\':\'v2: development chooses control/challenger; confirmation only accepts or rejects; custom challenger must beat both controls\'}\\n\\n\\ndef confirm_selection(plan,scores):\\n    \\"\\"\\"No best-of-confirmation search and no switching between novel families.\\"\\"\\"\\n    values={name:float(value[\'mcrmse\']) for name,value in scores.items()}\\n    required={\'table\',\'official\',plan[\'candidate\']}\\n    if not required.issubset(values) or not all(np.isfinite(list(values.values()))):\\n        raise ValueError(\'Missing or nonfinite confirmation score\')\\n    control=plan[\'control\']\\n    reasons=[]\\n    if control==\'official\' and not values[\'official\']<values[\'table\']-GATE_TOLERANCE:\\n        control=\'table\'\\n        reasons.append(\'Development-selected official control failed confirmation against table; retain table.\')\\n    retained=control\\n    candidate=plan[\'candidate\']\\n    if plan[\'challenger_eligible\']:\\n        beats_both=all(values[candidate]<values[name]-GATE_TOLERANCE for name in [\'table\',\'official\'])\\n        if beats_both:\\n            retained=candidate\\n            reasons.append(\'Frozen novel challenger passed both confirmation controls.\')\\n        else:\\n            reasons.append(\'Frozen novel challenger failed at least one control; reject it.\')\\n    else:\\n        reasons.append(\'No novel challenger beat the development-selected control; no novel confirmation search.\')\\n    return retained,{\'fallback_control\':control,\'reasons\':reasons}\\n\\n\\ndef run(root,output,cache=\'/kaggle/temp/lavalabs_features_v1.npz\'):\\n    started=time.monotonic()\\n    output=Path(output)\\n    output.mkdir(parents=True,exist_ok=True)\\n    train,test,sample,columns=load_tables(root)\\n    y=train[TARGETS].to_numpy(dtype=float)\\n    groups=train.site_key.to_numpy()\\n    ids=[*train.sample_id,*test.sample_id]\\n    print(\'Input contract passed:\',{\'train\':len(train),\'test\':len(test),\'sites\':len(set(groups))},flush=True)\\n    matrices,feature_receipt=feature_matrices(root,ids,cache)\\n    tx=train[columns].to_numpy(dtype=float)\\n    zx=test[columns].to_numpy(dtype=float)\\n    train_features={\'table\':tx,**{name:np.column_stack([tx,array[:len(train)]]) for name,array in matrices.items()}}\\n    test_features={\'table\':zx,**{name:np.column_stack([zx,array[len(train):]]) for name,array in matrices.items()}}\\n    # The confirmation sites are excluded from candidate/weight selection below.\\n    dev,confirmation=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=2026).split(tx,groups=groups))\\n    assert set(groups[dev]).isdisjoint(groups[confirmation])\\n    dev_features={name:x[dev] for name,x in train_features.items()}\\n    comparison=nested_compare(dev_features,y[dev],groups[dev],site_folds(groups[dev],3,seed=2026),inner_splits=3)\\n    scores=comparison[\'scores\']\\n    plan=selection_plan(scores)\\n    candidate=plan[\'candidate\']\\n    # Freeze the feature family before examining confirmation labels.\\n    selection={**plan,\'basis\':\'development-only control and single challenger selection; simpler families win numerical ties\',\\n               \'development_sites\':len(set(groups[dev])),\'confirmation_sites\':len(set(groups[confirmation])),\\n               \'development_comparison\':_public_summary(comparison)}\\n    write_json(output/\'selection_before_confirmation.json\',selection)\\n    print(\'Feature family frozen before confirmation:\',candidate,flush=True)\\n    confirmation_scores={}\\n    for name in dict.fromkeys([\'table\',\'official\',candidate]):\\n        pred,_=final_predict({k:v[dev] for k,v in train_features.items()},\\n                             {k:v[confirmation] for k,v in train_features.items()},y[dev],groups[dev],name,n_splits=3)\\n        confirmation_scores[name]=metric(y[confirmation],pred)\\n        print(\'Confirmation\',name,confirmation_scores[name][\'mcrmse\'],flush=True)\\n    # Confirmation is an accept/reject gate, so its selected result is not an untouched test estimate.\\n    retained,gate=confirm_selection(plan,confirmation_scores)\\n    prediction,choices=final_predict(train_features,test_features,y,groups,retained,n_splits=5)\\n    export_hash=export_submission(sample,prediction,output/\'submission_selected.csv\')\\n    official_prediction,official_choices=final_predict(train_features,test_features,y,groups,\'official\',n_splits=5)\\n    official_hash=export_submission(sample,official_prediction,output/\'submission_official_control.csv\')\\n    receipt={\'schema\':2,\'experiment\':\'cloud_spatial_nested_site_selection\',\\n             \'si_disclosure\':\'OpenAI Codex assisted code, tests, analysis implementation and documentation. Competition observations retain original provenance.\',\\n             \'counts\':{\'train\':len(train),\'test\':len(test),\'sites\':len(set(groups))},\\n             \'feature_extraction\':feature_receipt,\'selection\':selection,\'confirmation\':confirmation_scores,\\n             \'retained_variant\':retained,\'confirmation_gate\':gate,\'retained_choices\':choices,\'official_control_choices\':official_choices,\\n             \'parameters\':PARAMS,\'final_fit\':\'all permitted training rows with observed target masks; final weights from whole-site OOF\',\\n             \'environment\':{\'python\':platform.python_version(),**{p:importlib.metadata.version(p) for p in [\'numpy\',\'pandas\',\'scikit-learn\',\'rasterio\',\'scipy\']}},\\n             \'submissions\':{\'selected\':{\'rows\':len(test),\'sha256\':export_hash,\'file\':\'submission_selected.csv\'},\\n                            \'official_control\':{\'rows\':len(test),\'sha256\':official_hash,\'file\':\'submission_official_control.csv\'}},\\n             \'seconds\':round(time.monotonic()-started,3),\\n             \'limitations\':[\'Confirmation is used for an accept/reject gate; its selected result is not an untouched estimate.\',\\n                            \'Gate v2 was fixed during initial image extraction, before any image development/confirmation scores were observed; it also requires novel changes to beat the official control.\',\\n                            \'The earlier raw baseline aggregate evaluated all training sites before this refinement, including confirmation sites.\',\\n                            \'No true forward-time validation is possible with withheld dates. Private-test error remains unknown.\',\\n                            \'Feature-family selection uses development outer scores; those selected scores have model-selection optimism.\']}\\n    write_json(output/\'experiment_receipt.json\',receipt)\\n    print(\'EXPERIMENT_COMPLETE\',flush=True)\\n    print({\'development\':{name:value[\'mcrmse\'] for name,value in scores.items()},\'candidate\':candidate,\'retained\':retained,\\n           \'confirmation\':{name:value[\'mcrmse\'] for name,value in confirmation_scores.items()},\'seconds\':receipt[\'seconds\']},flush=True)\\n    return receipt\\n", "features.py": "\\"\\"\\"Content-only chip summaries; explicit cloud ablations, no geospatial metadata.\\n\\nThe official feature recipe is adapted from Zelzy\'s Apache-2.0 starter.\\nOpenAI Codex assisted the implementation and tests.\\n\\"\\"\\"\\nimport numpy as np\\n\\nBAND_ORDER = [\'B1\',\'B2\',\'B3\',\'B4\',\'B5\',\'B6\',\'B7\',\'B8\',\'B8A\',\'B9\',\'B11\',\'B12\',\'SCL\',\'CLDPRB\']\\nSTAT_BANDS = [\'B2\',\'B3\',\'B4\',\'B8\',\'B11\',\'B12\']\\nSCL_CLASSES = [2,3,4,5,6,7,8,9,10,11]\\nMIN_CLEAR = 500\\n\\n\\ndef _statistics(values, prefix):\\n    if not values.size:\\n        return {f\'{prefix}_{name}\': float(\'nan\') for name in [\'mean\',\'std\',\'p50\',\'p95\',\'p99\',\'max\']}\\n    p50,p95,p99 = np.percentile(values,[50,95,99])\\n    return {f\'{prefix}_{name}\':float(value) for name,value in\\n            zip([\'mean\',\'std\',\'p50\',\'p95\',\'p99\',\'max\'],[values.mean(),values.std(),p50,p95,p99,values.max()])}\\n\\n\\ndef _spatial(bands, mask, sufficient):\\n    names = [f\'s_{band}_{stat}\' for band in [\'B11\',\'B12\'] for stat in [\'centre_mean\',\'outer_mean\',\'centre_minus_outer\']]\\n    names += [\'s_ndvi_centre_minus_outer\',\'s_bsi_centre_minus_outer\',\'s_B12_tail_contrast\',\'s_B11_tail_contrast\',\\n              \'s_B12_centre_std\',\'s_B12_gradient\',\'s_B11_gradient\',\'s_ndvi_gradient\',\'s_centre_clear_fraction\',\'s_outer_clear_fraction\']\\n    if not sufficient:\\n        return dict.fromkeys(names,float(\'nan\'))\\n    height,width = mask.shape\\n    rows,cols = np.indices(mask.shape)\\n    centre = (rows>=height/4)&(rows<3*height/4)&(cols>=width/4)&(cols<3*width/4)\\n    c,edge = mask&centre,mask&~centre\\n    output = {}\\n    for band in [\'B11\',\'B12\']:\\n        a = bands[band]\\n        cm = float(a[c].mean()) if c.sum()>=20 else float(\'nan\')\\n        em = float(a[edge].mean()) if edge.sum()>=20 else float(\'nan\')\\n        output.update({f\'s_{band}_centre_mean\':cm,f\'s_{band}_outer_mean\':em,f\'s_{band}_centre_minus_outer\':cm-em})\\n    ndvi = (bands[\'B8\']-bands[\'B4\'])/(bands[\'B8\']+bands[\'B4\']+1e-6)\\n    bsi = ((bands[\'B11\']+bands[\'B4\'])-(bands[\'B8\']+bands[\'B2\']))/(bands[\'B11\']+bands[\'B4\']+bands[\'B8\']+bands[\'B2\']+1e-6)\\n    for label,a in [(\'ndvi\',ndvi),(\'bsi\',bsi)]:\\n        output[f\'s_{label}_centre_minus_outer\'] = float(a[c].mean()-a[edge].mean()) if min(c.sum(),edge.sum())>=20 else float(\'nan\')\\n    for band in [\'B12\',\'B11\']:\\n        p50,p99 = np.percentile(bands[band][mask],[50,99])\\n        output[f\'s_{band}_tail_contrast\'] = float(p99-p50)\\n    output[\'s_B12_centre_std\'] = float(bands[\'B12\'][c].std()) if c.sum()>=20 else float(\'nan\')\\n    for label,a in [(\'B12\',bands[\'B12\']),(\'B11\',bands[\'B11\']),(\'ndvi\',ndvi)]:\\n        vertical = np.abs(np.diff(a,axis=0))[mask[1:]&mask[:-1]]\\n        horizontal = np.abs(np.diff(a,axis=1))[mask[:,1:]&mask[:,:-1]]\\n        values = np.concatenate([vertical,horizontal])\\n        output[f\'s_{label}_gradient\'] = float(values.mean()) if values.size>=20 else float(\'nan\')\\n    output[\'s_centre_clear_fraction\'] = float(mask[centre].mean())\\n    output[\'s_outer_clear_fraction\'] = float(mask[~centre].mean())\\n    return output\\n\\n\\ndef chip_summary(chip, method=\'official\', spatial=False):\\n    chip = np.asarray(chip)\\n    if chip.ndim!=3 or chip.shape[0]!=14 or min(chip.shape[1:])<8:\\n        raise ValueError(\'Expected fourteen chip layers and at least eight pixels per dimension\')\\n    if method not in {\'official\',\'no_fallback\',\'strict\'}:\\n        raise ValueError(\'Unknown cloud-mask method\')\\n    if not np.isfinite(chip).all():\\n        raise ValueError(\'Unexpected nonfinite source chip\')\\n    scl,cloud = chip[12],chip[13]\\n    has_data = chip[3]>0\\n    clear = has_data&~np.isin(scl,[3,8,9,10])\\n    if method==\'strict\': clear = has_data&np.isin(scl,[4,5,6])&(cloud<20)\\n    sufficient = clear.sum()>=MIN_CLEAR\\n    mask = has_data if method==\'official\' and not sufficient else clear\\n    bands = {name:chip[index].astype(np.float64)*1e-4 for index,name in enumerate(BAND_ORDER[:12])}\\n    available = bool(mask.any()) if method==\'official\' else bool(sufficient)\\n    output = {}\\n    for band in STAT_BANDS:\\n        values = bands[band][mask] if available else np.array([])\\n        if method==\'official\' and not values.size: values=np.zeros(1)\\n        output.update(_statistics(values,f\'c_{band}\'))\\n    if available:\\n        b4,b8,b11,b12 = (bands[name][mask] for name in [\'B4\',\'B8\',\'B11\',\'B12\'])\\n        ndvi = (b8-b4)/(b8+b4+1e-6)\\n        values = [ndvi.std(),np.percentile(ndvi,95),np.percentile(b12/(b11+1e-6),95),\\n                  (b12>.2).mean(),(b12>.4).mean(),(b11>.4).mean()]\\n    else:\\n        values = [0.]*6 if method==\'official\' else [float(\'nan\')]*6\\n    output.update(dict(zip([\'c_ndvi_std\',\'c_ndvi_p95\',\'c_b12_b11_p95\',\'c_hot12_gt_0.2\',\'c_hot12_gt_0.4\',\'c_hot11_gt_0.4\'],map(float,values))))\\n    for cls in SCL_CLASSES: output[f\'c_scl_{cls}\']=float((scl==cls).mean())\\n    output[\'c_cloudprob\']=float(cloud.mean())\\n    output[\'c_valid_frac\']=float(mask.mean())\\n    if method!=\'official\':\\n        output[\'q_insufficient_clear\']=float(not sufficient)\\n        output[\'q_clear_count\']=float(clear.sum())\\n        output[\'q_cloud_high_fraction\']=float((cloud>=40).mean())\\n    if spatial: output.update(_spatial(bands,mask,available))\\n    return output\\n", "selection.py": "\\"\\"\\"Target-wise conservative mixtures with selection confined to inner sites.\\"\\"\\"\\nimport time\\nimport numpy as np\\nfrom .core import TARGETS, metric, site_folds\\nfrom .baseline import fit_predict\\n\\nAMPLITUDES = [0.,.25,.5,.75,1.]\\nIMAGE_WEIGHTS = [0.,.5,1.]\\n\\n\\ndef mix_predictions(base, enhanced, mean, amplitude, image_weight):\\n    if not (0<=amplitude<=1 and 0<=image_weight<=1):\\n        raise ValueError(\'Only convex prediction mixtures are permitted\')\\n    base,enhanced,mean = map(np.asarray,[base,enhanced,mean])\\n    return mean+amplitude*((1-image_weight)*(base-mean)+image_weight*(enhanced-mean))\\n\\n\\ndef choose_mix(truth, base, enhanced, means, image_weights=None):\\n    y,b,e,m = map(lambda a:np.asarray(a,dtype=float),[truth,base,enhanced,means])\\n    if any(a.shape!=y.shape for a in [b,e,m]) or not all(np.isfinite(a).all() for a in [b,e,m]):\\n        raise ValueError(\'Mixture inputs must have matching shapes and finite predictions\')\\n    keep = np.isfinite(y)\\n    if np.isinf(y).any() or keep.sum()<2:\\n        raise ValueError(\'Need at least two measured, finite tuning labels\')\\n    best=None\\n    for w in (IMAGE_WEIGHTS if image_weights is None else image_weights):\\n        for a in AMPLITUDES:\\n            p=mix_predictions(b,e,m,a,w)\\n            error=float(np.sqrt(np.mean((y[keep]-p[keep])**2)))\\n            key=(error,w,a)\\n            if best is None or error<best[0][0]-1e-12 or (abs(error-best[0][0])<=1e-12 and (w,a)<best[0][1:]):\\n                best=(key,{\'amplitude\':a,\'image_weight\':w,\'inner_rmse\':error})\\n    return best[1]\\n\\n\\ndef raw_oof(x,y,groups,n_splits=3,params=None):\\n    y=np.asarray(y,dtype=float)\\n    predictions=np.full(y.shape,np.nan)\\n    means=np.full(y.shape,np.nan)\\n    for fit,check in site_folds(groups,n_splits):\\n        for j in range(5):\\n            _,predictions[check,j]=fit_predict(x[fit],y[fit,j],x[check],params)\\n            means[check,j]=np.nanmean(y[fit,j])\\n    if not np.isfinite(predictions).all() or not np.isfinite(means).all():\\n        raise AssertionError(\'Incomplete inner cross-validation\')\\n    return predictions,means\\n\\n\\ndef nested_compare(features,truth,groups,folds=None,inner_splits=3,params=None):\\n    y=np.asarray(truth,dtype=float)\\n    groups=np.asarray(groups)\\n    features={key:np.asarray(value,dtype=float) for key,value in features.items()}\\n    if \'table\' not in features or any(len(value)!=len(y) for value in features.values()):\\n        raise ValueError(\'Features require the shared table reference and matching rows\')\\n    folds=site_folds(groups,3) if folds is None else folds\\n    predictions={key:np.full(y.shape,np.nan) for key in features}\\n    raw={key:np.full(y.shape,np.nan) for key in features}\\n    means=np.full(y.shape,np.nan)\\n    reports=[]\\n    seen=np.zeros(len(y),dtype=int)\\n    for number,(fit,check) in enumerate(folds,1):\\n        tick=time.monotonic()\\n        if set(groups[fit])&set(groups[check]) or np.intersect1d(fit,check).size:\\n            raise ValueError(\'Outer facility leakage\')\\n        seen[check]+=1\\n        inner={}\\n        for name,x in features.items():\\n            inner[name],inner_means=raw_oof(x[fit],y[fit],groups[fit],inner_splits,params)\\n        for name,x in features.items():\\n            for j in range(5):\\n                _,raw[name][check,j]=fit_predict(x[fit],y[fit,j],x[check],params)\\n        means[check]=np.nanmean(y[fit],axis=0)\\n        choices={}\\n        for name in features:\\n            choices[name]=[]\\n            for j in range(5):\\n                choice=choose_mix(y[fit,j],inner[\'table\'][:,j],inner[name][:,j],inner_means[:,j],\\n                                  image_weights=[0.] if name==\'table\' else None)\\n                choices[name].append(choice)\\n                predictions[name][check,j]=mix_predictions(raw[\'table\'][check,j],raw[name][check,j],\\n                    means[check,j],choice[\'amplitude\'],choice[\'image_weight\'])\\n        reports.append({\'fold\':number,\'fit_sites\':len(set(groups[fit])),\'check_sites\':len(set(groups[check])),\\n                        \'choices\':choices,\'scores\':{name:metric(y[check],p[check]) for name,p in predictions.items()},\\n                        \'seconds\':round(time.monotonic()-tick,3)})\\n        print(\'Outer fold\',number,\'scores:\',{name:round(value[\'mcrmse\'],6) for name,value in reports[-1][\'scores\'].items()},flush=True)\\n    if not np.all(seen==1):\\n        raise ValueError(\'Outer folds must hold every row out exactly once\')\\n    scores={name:metric(y,p) for name,p in predictions.items()}\\n    return {\'scores\':scores,\'folds\':reports,\'predictions\':predictions,\'raw_predictions\':raw,\\n            \'mean_predictions\':means,\'fold_mean_score\':metric(y,means),\\n            \'raw_scores\':{name:metric(y,p) for name,p in raw.items()}}\\n\\n\\ndef final_predict(train_features,test_features,truth,groups,variant,n_splits=5,params=None):\\n    y=np.asarray(truth,dtype=float)\\n    names=[\'table\'] if variant==\'table\' else [\'table\',variant]\\n    oof={}\\n    test_predictions={}\\n    for name in names:\\n        oof[name],means=raw_oof(np.asarray(train_features[name]),y,groups,n_splits,params)\\n        test_predictions[name]=np.zeros((len(test_features[name]),5))\\n        for j in range(5):\\n            _,test_predictions[name][:,j]=fit_predict(train_features[name],y[:,j],test_features[name],params)\\n    mu=np.nanmean(y,axis=0)\\n    result=np.zeros_like(test_predictions[\'table\'])\\n    choices=[]\\n    for j in range(5):\\n        choice=choose_mix(y[:,j],oof[\'table\'][:,j],oof[variant][:,j],means[:,j],\\n                          image_weights=[0.] if variant==\'table\' else None)\\n        result[:,j]=mix_predictions(test_predictions[\'table\'][:,j],test_predictions[variant][:,j],\\n                                    mu[j],choice[\'amplitude\'],choice[\'image_weight\'])\\n        choices.append(choice)\\n    if not np.isfinite(result).all(): raise ValueError(\'Nonfinite final prediction\')\\n    return result,choices\\n"}')
for name, text in sources.items():
    (package/name).write_text(text, encoding='utf-8')
for name in list(sys.modules):
    if name == 'atmospheric_lab' or name.startswith('atmospheric_lab.'):
        del sys.modules[name]
print('LavaLabs source ready; aggregate reporting only.')


In [ ]:
import numpy as np
from atmospheric_lab.core import metric, site_folds
y = np.array([[1,2,3,4,5],[np.nan,4,5,6,7]], dtype=float)
assert abs(metric(y,np.zeros_like(y))['mcrmse'] - np.mean([1,np.sqrt(10),np.sqrt(17),np.sqrt(26),np.sqrt(37)])) < 1e-12
groups = np.repeat(np.arange(6),3)
assert all(set(groups[a]).isdisjoint(groups[b]) for a,b in site_folds(groups,3))
print('Masked metric and whole-site split assertions passed.')


In [ ]:
RUN_IN_PRIVATE_COPY = False
if not RUN_IN_PRIVATE_COPY:
    print('Source-only release: create a private copy, attach OBSCURA input, and set RUN_IN_PRIVATE_COPY=True to reproduce. Do not publish row outputs.')
else:
    from pathlib import Path
    from atmospheric_lab.experiment import run
    roots = [Path('/kaggle/input/competitions/obscura'),Path('/kaggle/input/obscura')]
    data = next((p for p in roots if (p/'train.csv').exists()),None)
    if data is None:
        raise RuntimeError('Attach the official OBSCURA competition input before running.')
    receipt = run(data, '/kaggle/working/lavalabs_private_experiment')
    print('Private CSV ready. Do not publish row-level outputs.')
    print('AGGREGATE_RECEIPT_BEGIN')
    print(json.dumps(receipt, sort_keys=True))
    print('AGGREGATE_RECEIPT_END')


## Upstream license

                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright [yyyy] [name of copyright owner]

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.


## Fixed CatBoost follow-up: separate frozen recipe
Same 83 inputs; fixed Ordered CPU configuration. Saved private run 372977582, 90 fits, development MCRMSE 0.931200 versus 0.933783 for HistGB, wins 2/3 folds. Reused development, not independent confirmation. Public 1.16407 versus HistGB 1.15808. Private unknown.
Keep both private-copy run flags False for this public source artifact. For the CatBoost recipe, execute the following bootstrap then enable only RUN_CAT_IN_PRIVATE_COPY in your private copy with competition input attached.


In [ ]:
from pathlib import Path
import json, os, sys
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
package = Path('/kaggle/working/atmospheric_lab')
package.mkdir(exist_ok=True)
sources = json.loads('{"__init__.py": "\\"\\"\\"LavaLabs Atmospheric Lab. SI-assisted code; scientific inputs retain provenance.\\"\\"\\"\\n", "baseline.py": "\\"\\"\\"First measured baseline. Official HistGB recipe, supplied features, no selection.\\"\\"\\"\\nfrom pathlib import Path\\nimport importlib.metadata\\nimport platform\\nimport time\\nimport numpy as np\\nfrom sklearn.ensemble import HistGradientBoostingRegressor\\nfrom threadpoolctl import threadpool_limits\\nfrom .core import TARGETS, SOURCE, load_tables, site_folds, metric, export_submission, write_json\\n\\nPARAMS = dict(max_iter=250, learning_rate=.04, max_depth=3,\\n              min_samples_leaf=40, l2_regularization=5., max_features=.6,\\n              random_state=0, early_stopping=False)\\n\\n\\ndef fit_predict(x, y, z, params=None):\\n    y = np.asarray(y, dtype=float)\\n    keep = np.isfinite(y)\\n    if keep.sum() < 2:\\n        raise ValueError(\'Not enough measured labels in the fitting partition\')\\n    with threadpool_limits(limits=2):\\n        model = HistGradientBoostingRegressor(**(PARAMS if params is None else params))\\n        model.fit(np.asarray(x)[keep], y[keep])\\n        prediction = model.predict(z)\\n    if not np.isfinite(prediction).all():\\n        raise ValueError(\'Estimator produced a nonfinite prediction\')\\n    return model, prediction\\n\\n\\ndef run(root, output, n_splits=5, params=None, expected_rows=1221):\\n    started = time.monotonic()\\n    output = Path(output)\\n    output.mkdir(parents=True, exist_ok=True)\\n    train, test, sample, names = load_tables(root, expected_rows)\\n    x = train[names].to_numpy(dtype=float)\\n    z = test[names].to_numpy(dtype=float)\\n    y = train[TARGETS].to_numpy(dtype=float)\\n    folds = site_folds(train.site_key.to_numpy(), n_splits)\\n    oof = np.full(y.shape, np.nan)\\n    mean_oof = np.full(y.shape, np.nan)\\n    fold_reports = []\\n    for fold_number, (fit, check) in enumerate(folds, 1):\\n        tick = time.monotonic()\\n        for j in range(5):\\n            _, oof[check, j] = fit_predict(x[fit], y[fit, j], x[check], params)\\n            mean_oof[check, j] = np.nanmean(y[fit, j])\\n        report = metric(y[check], oof[check])\\n        report.update(fold=fold_number, seconds=round(time.monotonic()-tick, 3),\\n                      heldout_sites=int(train.iloc[check].site_key.nunique()),\\n                      fit_sites=int(train.iloc[fit].site_key.nunique()))\\n        fold_reports.append(report)\\n        print(f\'Fold {fold_number}/{n_splits}: MCRMSE {report[\\"mcrmse\\"]:.6f}; {report[\\"seconds\\"]:.1f}s\', flush=True)\\n    predictions = np.zeros((len(test), 5))\\n    for j in range(5):\\n        _, predictions[:, j] = fit_predict(x, y[:, j], z, params)\\n    export_hash = export_submission(sample, predictions, output/\'submission_table.csv\')\\n    receipt = {\'schema\': 1, \'experiment\': \'supplied_table_unselected_histgb\',\\n               \'si_disclosure\': \'OpenAI Codex assisted code, tests and documentation; source observations retain competition provenance.\',\\n               \'baseline_reference\': SOURCE, \'baseline_adaptation\': \'29 supplied features; explicit no early stopping; no OOF-tuned shrinkage.\',\\n               \'counts\': {\'train\': len(train), \'test\': len(test), \'sites\': int(train.site_key.nunique()), \'features\': len(names)},\\n               \'validation\': {\'method\': \'whole-site GroupKFold\', \'folds\': n_splits, \'site_overlap\': False,\\n                              \'selection\': \'none; fixed baseline parameters\', \'pooled_model\': metric(y, oof),\\n                              \'pooled_fold_mean\': metric(y, mean_oof), \'pooled_zero\': metric(y, np.zeros_like(y)),\\n                              \'fold_reports\': fold_reports},\\n               \'parameters\': PARAMS if params is None else params, \'seconds\': round(time.monotonic()-started, 3),\\n               \'environment\': {\'python\': platform.python_version(), **{p: importlib.metadata.version(p) for p in [\'numpy\',\'pandas\',\'scikit-learn\',\'scipy\']}},\\n               \'submission\': {\'rows\': len(test), \'sha256\': export_hash, \'finite\': True, \'private_file\': \'submission_table.csv\'},\\n               \'boundary\': \'Unseen-site validation within training time; private/future-time performance is unknown. No leaderboard score yet.\'}\\n    write_json(output/\'baseline_receipt.json\', receipt)\\n    print(\'BASELINE_COMPLETE\', flush=True)\\n    print(receipt[\'validation\'][\'pooled_model\'], flush=True)\\n    return receipt\\n", "cat_followup.py": "\\"\\"\\"One predeclared Ordered CatBoost comparison. No leaderboard parameter search.\\"\\"\\"\\nfrom pathlib import Path\\nimport importlib.metadata\\nimport time\\nimport numpy as np\\nfrom sklearn.model_selection import GroupShuffleSplit\\nfrom .core import TARGETS,load_tables,site_folds,write_json,export_submission\\nfrom .experiment import feature_matrices,_public_summary\\nfrom .selection import nested_compare,final_predict\\n\\nCAT_PARAMS=dict(loss_function=\'RMSE\',iterations=400,learning_rate=.03,depth=3,\\n    grow_policy=\'SymmetricTree\',boosting_type=\'Ordered\',l2_leaf_reg=10.,\\n    bootstrap_type=\'Bernoulli\',subsample=.8,sampling_frequency=\'PerTree\',rsm=.8,\\n    random_strength=1.,random_seed=2026,nan_mode=\'Min\',boost_from_average=True,\\n    use_best_model=False,task_type=\'CPU\',thread_count=2,verbose=False,allow_writing_files=False)\\n\\n\\ndef fit_cat(x,y,z,params=None):\\n    from catboost import CatBoostRegressor\\n    x,y,z=map(np.asarray,[x,y,z])\\n    if x.ndim!=2 or z.ndim!=2 or y.ndim!=1 or len(x)!=len(y) or x.shape[1]!=z.shape[1]:\\n        raise ValueError(\'CatBoost inputs must be numeric matrices and one target vector\')\\n    if any(a.dtype.kind not in \'fiu\' for a in [x,y,z]) or any(np.isinf(a).any() for a in [x,y,z]):\\n        raise ValueError(\'CatBoost inputs must be numeric and cannot contain infinity\')\\n    keep=np.isfinite(y)\\n    if keep.sum()<2:raise ValueError(\'Not enough observed CatBoost fitting labels\')\\n    x,y,z=x.astype(float),y.astype(float),z.astype(float)\\n    if np.ptp(y[keep])==0:\\n        return None,np.full(len(z),float(y[keep][0]))\\n    model=CatBoostRegressor(**(CAT_PARAMS if params is None else params))\\n    model.fit(x[keep],y[keep])\\n    if model.tree_count_!=(CAT_PARAMS if params is None else params)[\'iterations\']:\\n        raise AssertionError(\'Unexpected CatBoost iteration selection\')\\n    prediction=model.predict(z)\\n    if not np.isfinite(prediction).all():raise ValueError(\'Nonfinite CatBoost prediction\')\\n    return model,prediction\\n\\n\\ndef run(root,output,cache=\'/kaggle/temp/lavalabs_features_v1.npz\'):\\n    started=time.monotonic();output=Path(output);output.mkdir(parents=True,exist_ok=True)\\n    train,test,sample,columns=load_tables(root)\\n    x=train[columns].to_numpy(dtype=float);z=test[columns].to_numpy(dtype=float)\\n    y=train[TARGETS].to_numpy(dtype=float);groups=train.site_key.to_numpy()\\n    matrices,features=feature_matrices(root,[*train.sample_id,*test.sample_id],cache)\\n    all_features={\'table\':x,\'official\':np.column_stack([x,matrices[\'official\'][:len(x)]])}\\n    test_features={\'table\':z,\'official\':np.column_stack([z,matrices[\'official\'][len(x):]])}\\n    all_features[\'cat\']=all_features[\'official\'];test_features[\'cat\']=test_features[\'official\']\\n    dev,_=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=2026).split(x,groups=groups))\\n    counter={\'fits\':0,\'seconds\':[],\'resolved\':None}\\n    def bounded_fit(a,b,c,params=None):\\n        if counter[\'fits\']>=90 or time.monotonic()-started>7200:\\n            raise RuntimeError(\'Fixed CatBoost experiment exceeded its 90-fit / two-hour cap; retain v2.\')\\n        tick=time.monotonic();model,pred=fit_cat(a,b,c)\\n        counter[\'fits\']+=1;counter[\'seconds\'].append(time.monotonic()-tick)\\n        if model is not None and counter[\'resolved\'] is None:counter[\'resolved\']=model.get_all_params()\\n        if counter[\'fits\'] in [1,2]:\\n            estimated=np.mean(counter[\'seconds\'])*(90-counter[\'fits\'])\\n            print(\'CatBoost timed fit\',counter[\'fits\'],round(counter[\'seconds\'][-1],3),\'seconds; remaining fit projection\',round(estimated,1),flush=True)\\n            if estimated+time.monotonic()-started>7200:\\n                raise RuntimeError(\'Fixed CatBoost fit timing exceeds two-hour cap; retain v2 without retuning.\')\\n        return model,pred\\n    comparison=nested_compare({k:v[dev] for k,v in all_features.items()},y[dev],groups[dev],\\n        site_folds(groups[dev],3,seed=2026),inner_splits=3,fitters={\'cat\':bounded_fit})\\n    control=comparison[\'scores\'][\'official\'][\'mcrmse\'];candidate=comparison[\'scores\'][\'cat\'][\'mcrmse\']\\n    fold_wins=sum(f[\'scores\'][\'cat\'][\'mcrmse\']<f[\'scores\'][\'official\'][\'mcrmse\']-1e-10 for f in comparison[\'folds\'])\\n    retained=\'cat\' if candidate<control-1e-10 and fold_wins>=2 else \'official\'\\n    receipt={\'schema\':1,\'experiment\':\'one_fixed_ordered_catboost\',\'parameters\':CAT_PARAMS,\\n      \'catboost_version\':importlib.metadata.version(\'catboost\'),\'comparison\':_public_summary(comparison),\\n      \'decision\':{\'retained\':retained,\'cat_fold_wins\':fold_wins,\'rule\':\'Cat must improve pooled development MCRMSE by >1e-10 and win at least two of three paired outer folds; otherwise retain v2 official.\'},\\n      \'counts\':{\'train\':len(train),\'test\':len(test),\'sites\':len(set(groups)),\'development_sites\':len(set(groups[dev]))},\\n      \'limitations\':[\'This is iterative development on previously used sites, not new independent confirmation.\',\'The previous confirmation sites are not scored again for this comparison.\',\'Private/future-time transfer remains unknown. No public-score fitting.\'],\\n      \'feature_extraction\':features,\'si_disclosure\':\'OpenAI Codex assisted code, tests and analysis. Original competition observations unchanged.\'}\\n    write_json(output/\'selection_before_refit.json\',receipt)\\n    if retained==\'cat\':\\n        pred,choices=final_predict(all_features,test_features,y,groups,\'cat\',n_splits=5,fitters={\'cat\':bounded_fit})\\n        digest=export_submission(sample,pred,output/\'submission_cat.csv\')\\n        receipt.update(final_choices=choices,submission={\'rows\':len(test),\'sha256\':digest,\'file\':\'submission_cat.csv\'})\\n    receipt.update(cat_fits=counter[\'fits\'],cat_fit_seconds=counter[\'seconds\'],resolved_catboost_parameters=counter[\'resolved\'],seconds=time.monotonic()-started)\\n    write_json(output/\'catboost_receipt.json\',receipt)\\n    print(\'FIXED_CATBOOST_COMPLETE\',{\'retained\':retained,\'control\':control,\'candidate\':candidate,\'fold_wins\':fold_wins,\'seconds\':round(receipt[\'seconds\'],1)},flush=True)\\n    return receipt\\n", "core.py": "\\"\\"\\"Contracts and numerical checks for masked, whole-site atmospheric regression.\\"\\"\\"\\nfrom pathlib import Path\\nimport hashlib\\nimport json\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.model_selection import GroupKFold\\n\\nTARGETS = [\'t_ch4\', \'t_no2\', \'t_co\', \'t_aai\', \'t_firms\']\\nSOURCE = \'https://www.kaggle.com/code/zelza0/obscura-starter-baseline\'\\n\\n\\ndef metric(truth, prediction):\\n    \\"\\"\\"Official pooled, masked target RMSEs; never ignore a bad prediction.\\"\\"\\"\\n    y, p = np.asarray(truth, dtype=float), np.asarray(prediction, dtype=float)\\n    if y.ndim != 2 or y.shape != p.shape or y.shape[1] != 5:\\n        raise ValueError(\'Expected equally shaped n-by-five targets and predictions\')\\n    if np.isinf(y).any() or not np.isfinite(p).all():\\n        raise ValueError(\'Targets cannot be infinite; every prediction must be finite\')\\n    present = np.isfinite(y)\\n    counts = present.sum(axis=0)\\n    if (counts == 0).any():\\n        raise ValueError(\'All five targets need at least one observed value to report MCRMSE\')\\n    errors = [float(np.sqrt(np.mean((y[present[:, j], j] - p[present[:, j], j]) ** 2)))\\n              for j in range(5)]\\n    return {\'mcrmse\': float(np.mean(errors)), \'rmse\': dict(zip(TARGETS, errors)),\\n            \'observed\': dict(zip(TARGETS, map(int, counts)))}\\n\\n\\ndef validate_tables(train, test, sample, expected_rows=None):\\n    for name, frame in [(\'train\', train), (\'test\', test), (\'sample\', sample)]:\\n        if frame.columns.duplicated().any() or \'sample_id\' not in frame:\\n            raise ValueError(f\'{name}: duplicate columns or missing identifier\')\\n        ids = frame[\'sample_id\']\\n        if ids.isna().any() or not ids.is_unique or not ids.map(lambda x: isinstance(x, str) and bool(x.strip())).all():\\n            raise ValueError(f\'{name}: identifiers must be unique, nonempty strings\')\\n    if set(train.sample_id) & set(test.sample_id):\\n        raise ValueError(\'Training and test sample IDs overlap\')\\n    if list(sample.columns) != [\'sample_id\', *TARGETS]:\\n        raise ValueError(\'Submission template columns do not match the contract\')\\n    if list(test.sample_id) != list(sample.sample_id):\\n        raise ValueError(\'Test and template identifier order differs\')\\n    if expected_rows is not None and len(test) != expected_rows:\\n        raise ValueError(\'Unexpected test row count\')\\n    if \'site_key\' not in train or train.site_key.isna().any():\\n        raise ValueError(\'Training sites must be present\')\\n    features = [c for c in test if c != \'sample_id\']\\n    if any(c in features for c in [\'site_key\', *TARGETS]):\\n        raise ValueError(\'Label or site identifier found among predictors\')\\n    if set(train.columns) != {\'sample_id\', \'site_key\', *TARGETS, *features}:\\n        raise ValueError(\'Training and test feature schemas differ\')\\n    for frame, names in [(train, [*features, *TARGETS]), (test, features)]:\\n        for name in names:\\n            if not pd.api.types.is_numeric_dtype(frame[name]) or np.isinf(frame[name].to_numpy(dtype=float)).any():\\n                raise ValueError(\'Predictors and targets must be numerical, missing or finite\')\\n    if train[TARGETS].notna().sum().eq(0).any():\\n        raise ValueError(\'A training target has no observations\')\\n    return features\\n\\n\\ndef load_tables(root, expected_rows=1221):\\n    root = Path(root)\\n    train = pd.read_csv(root / \'train.csv\', dtype={\'sample_id\': str, \'site_key\': str})\\n    test = pd.read_csv(root / \'test.csv\', dtype={\'sample_id\': str})\\n    sample = pd.read_csv(root / \'sample_submission.csv\', dtype={\'sample_id\': str})\\n    features = validate_tables(train, test, sample, expected_rows)\\n    if len(features) != 29:\\n        raise ValueError(\'Expected the official 29 supplied numerical features\')\\n    return train, test, sample, features\\n\\n\\ndef site_folds(groups, n_splits=5, seed=None):\\n    groups = np.asarray(groups)\\n    if pd.isna(groups).any() or len(set(groups)) < n_splits:\\n        raise ValueError(\'Not enough valid sites for the requested folds\')\\n    splitter = GroupKFold(n_splits=n_splits) if seed is None else GroupKFold(n_splits=n_splits,shuffle=True,random_state=seed)\\n    folds = list(splitter.split(np.zeros(len(groups)), groups=groups))\\n    seen = np.zeros(len(groups), dtype=int)\\n    for fit, check in folds:\\n        if set(groups[fit]) & set(groups[check]):\\n            raise AssertionError(\'Site leakage across a fold\')\\n        seen[check] += 1\\n    if not np.all(seen == 1):\\n        raise AssertionError(\'Every row must be held out exactly once\')\\n    return folds\\n\\n\\ndef export_submission(sample, predictions, destination):\\n    p = np.asarray(predictions, dtype=float)\\n    if p.shape != (len(sample), 5) or not np.isfinite(p).all():\\n        raise ValueError(\'Test predictions must be finite and match all template rows\')\\n    if not sample.sample_id.is_unique or not sample.sample_id.map(lambda x: isinstance(x, str)).all():\\n        raise ValueError(\'Invalid submission identifiers\')\\n    result = sample[[\'sample_id\']].copy()\\n    result[TARGETS] = p\\n    path = Path(destination)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    result.to_csv(path, index=False, float_format=\'%.9g\')\\n    replay = pd.read_csv(path, dtype={\'sample_id\': str})\\n    if list(replay.sample_id) != list(sample.sample_id) or not np.isfinite(replay[TARGETS]).all().all():\\n        raise AssertionError(\'Export round trip failed\')\\n    return hashlib.sha256(path.read_bytes()).hexdigest()\\n\\n\\ndef write_json(path, value):\\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\\\n\', encoding=\'utf-8\')\\n", "experiment.py": "\\"\\"\\"Bounded content-feature experiments and site-held-out confirmation on Kaggle.\\"\\"\\"\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom pathlib import Path\\nimport hashlib\\nimport importlib.metadata\\nimport inspect\\nimport json\\nimport platform\\nimport time\\nimport numpy as np\\nfrom sklearn.model_selection import GroupShuffleSplit\\nfrom . import features as feature_module\\nfrom .features import chip_summary, BAND_ORDER, _spatial\\nfrom .baseline import PARAMS\\nfrom .core import TARGETS, load_tables, site_folds, metric, write_json, export_submission\\nfrom .selection import nested_compare, final_predict\\n\\n\\ndef extract_one(path):\\n    import rasterio\\n    with rasterio.open(path) as source:\\n        chip=source.read()  # Intentionally do not inspect geospatial/file metadata.\\n    official=chip_summary(chip,\'official\')\\n    clear=(chip[3]>0)&~np.isin(chip[12],[3,8,9,10])\\n    sufficient=clear.sum()>=500\\n    if sufficient:\\n        revised=dict(official)\\n        revised.update(q_insufficient_clear=0.,q_clear_count=float(clear.sum()),q_cloud_high_fraction=float((chip[13]>=40).mean()))\\n    else:\\n        revised=chip_summary(chip,\'no_fallback\')\\n    bands={name:chip[index].astype(float)*1e-4 for index,name in enumerate(BAND_ORDER[:12])}\\n    spatial={**revised,**_spatial(bands,clear,bool(sufficient))}\\n    return official,revised,spatial\\n\\n\\ndef feature_matrices(root,ids,cache_path,workers=2):\\n    root,cache_path=Path(root),Path(cache_path)\\n    # Both functions define extraction; changing either must invalidate the cache.\\n    source_hash=hashlib.sha256((inspect.getsource(feature_module)+\'\\\\n\'+inspect.getsource(extract_one)).encode()).hexdigest()\\n    fingerprint=hashlib.sha256((source_hash+\'\\\\n\'+str(root.resolve())+\'\\\\n\'+\'\\\\n\'.join(ids)).encode()).hexdigest()\\n    if cache_path.exists():\\n        with np.load(cache_path,allow_pickle=False) as saved:\\n            if saved[\'fingerprint\'].item()==fingerprint:\\n                summary=json.loads(saved[\'summary\'].item())\\n                expected={\'official\':54,\'no_fallback\':57,\'spatial\':73}\\n                for name,width in expected.items():\\n                    value=saved[name]\\n                    names=summary[\'column_names\'][name]\\n                    if (value.shape!=(len(ids),width) or np.isinf(value).any()\\n                            or len(names)!=width or names!=sorted(set(names))\\n                            or summary[\'columns\'][name]!=width or summary[\'chip_count\']!=len(ids)\\n                            or summary[\'feature_source_sha256\']!=source_hash):\\n                        raise ValueError(\'Private feature cache failed schema/integrity checks; rebuild it.\')\\n                print(\'Verified private feature cache loaded.\',flush=True)\\n                return {name:saved[name].copy() for name in expected},summary\\n    rows={name:[] for name in [\'official\',\'no_fallback\',\'spatial\']}\\n    names={}\\n    started=time.monotonic()\\n    def worker(sid):\\n        try: return extract_one(root/\'images\'/f\'{sid}.tif\')\\n        except Exception:\\n            # Keep row identifiers and original path errors out of notebook outputs.\\n            raise RuntimeError(\'A competition image failed extraction; no fabricated fallback is permitted.\') from None\\n    with ThreadPoolExecutor(max_workers=workers) as pool:\\n        for number,triplet in enumerate(pool.map(worker,ids),1):\\n            for name,features in zip(rows,triplet):\\n                keys=sorted(features)\\n                if name not in names: names[name]=keys\\n                if keys!=names[name]: raise AssertionError(\'Feature schema changed across chips\')\\n                rows[name].append([features[k] for k in keys])\\n            if number%250==0 or number==len(ids):\\n                elapsed=time.monotonic()-started\\n                print(f\'Chips {number}/{len(ids)}; {elapsed:.1f}s; estimated {(len(ids)-number)*elapsed/max(number,1)/60:.1f} min remaining\',flush=True)\\n    matrices={name:np.asarray(value,dtype=np.float64) for name,value in rows.items()}\\n    for value in matrices.values():\\n        if np.isinf(value).any(): raise ValueError(\'Infinite extracted predictor\')\\n    revised=matrices[\'no_fallback\']\\n    summary={\'chip_count\':len(ids),\'columns\':{name:len(keys) for name,keys in names.items()},\\n             \'column_names\':names,\'insufficient_clear_fraction\':float(revised[:,names[\'no_fallback\'].index(\'q_insufficient_clear\')].mean()),\\n             \'seconds\':round(time.monotonic()-started,3),\'feature_source_sha256\':source_hash,\\n             \'boundary\':\'Relative pixels only; no geographic location, ground scale or outside observations.\'}\\n    cache_path.parent.mkdir(parents=True,exist_ok=True)\\n    np.savez_compressed(cache_path,**matrices,fingerprint=np.array(fingerprint),summary=np.array(json.dumps(summary)))\\n    return matrices,summary\\n\\n\\ndef _public_summary(comparison):\\n    return {key:comparison[key] for key in [\'scores\',\'folds\',\'fold_mean_score\',\'raw_scores\']}\\n\\n\\nGATE_TOLERANCE = 1e-10\\n\\n\\ndef selection_plan(scores):\\n    \\"\\"\\"Freeze one comparator and at most one novel challenger on development only.\\"\\"\\"\\n    values={name:float(value[\'mcrmse\']) for name,value in scores.items()}\\n    if set(values)!={\'table\',\'official\',\'no_fallback\',\'spatial\'} or not all(np.isfinite(list(values.values()))):\\n        raise ValueError(\'Need four finite development scores\')\\n    control=\'official\' if values[\'official\']<values[\'table\']-GATE_TOLERANCE else \'table\'\\n    novel=\'spatial\' if values[\'spatial\']<values[\'no_fallback\']-GATE_TOLERANCE else \'no_fallback\'\\n    eligible=values[novel]<values[control]-GATE_TOLERANCE\\n    return {\'control\':control,\'novel\':novel,\'challenger_eligible\':bool(eligible),\\n            \'candidate\':novel if eligible else control,\'tolerance\':GATE_TOLERANCE,\\n            \'gate_protocol\':\'v2: development chooses control/challenger; confirmation only accepts or rejects; custom challenger must beat both controls\'}\\n\\n\\ndef confirm_selection(plan,scores):\\n    \\"\\"\\"No best-of-confirmation search and no switching between novel families.\\"\\"\\"\\n    values={name:float(value[\'mcrmse\']) for name,value in scores.items()}\\n    required={\'table\',\'official\',plan[\'candidate\']}\\n    if not required.issubset(values) or not all(np.isfinite(list(values.values()))):\\n        raise ValueError(\'Missing or nonfinite confirmation score\')\\n    control=plan[\'control\']\\n    reasons=[]\\n    if control==\'official\' and not values[\'official\']<values[\'table\']-GATE_TOLERANCE:\\n        control=\'table\'\\n        reasons.append(\'Development-selected official control failed confirmation against table; retain table.\')\\n    retained=control\\n    candidate=plan[\'candidate\']\\n    if plan[\'challenger_eligible\']:\\n        beats_both=all(values[candidate]<values[name]-GATE_TOLERANCE for name in [\'table\',\'official\'])\\n        if beats_both:\\n            retained=candidate\\n            reasons.append(\'Frozen novel challenger passed both confirmation controls.\')\\n        else:\\n            reasons.append(\'Frozen novel challenger failed at least one control; reject it.\')\\n    else:\\n        reasons.append(\'No novel challenger beat the development-selected control; no novel confirmation search.\')\\n    return retained,{\'fallback_control\':control,\'reasons\':reasons}\\n\\n\\ndef run(root,output,cache=\'/kaggle/temp/lavalabs_features_v1.npz\'):\\n    started=time.monotonic()\\n    output=Path(output)\\n    output.mkdir(parents=True,exist_ok=True)\\n    train,test,sample,columns=load_tables(root)\\n    y=train[TARGETS].to_numpy(dtype=float)\\n    groups=train.site_key.to_numpy()\\n    ids=[*train.sample_id,*test.sample_id]\\n    print(\'Input contract passed:\',{\'train\':len(train),\'test\':len(test),\'sites\':len(set(groups))},flush=True)\\n    matrices,feature_receipt=feature_matrices(root,ids,cache)\\n    tx=train[columns].to_numpy(dtype=float)\\n    zx=test[columns].to_numpy(dtype=float)\\n    train_features={\'table\':tx,**{name:np.column_stack([tx,array[:len(train)]]) for name,array in matrices.items()}}\\n    test_features={\'table\':zx,**{name:np.column_stack([zx,array[len(train):]]) for name,array in matrices.items()}}\\n    # The confirmation sites are excluded from candidate/weight selection below.\\n    dev,confirmation=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=2026).split(tx,groups=groups))\\n    assert set(groups[dev]).isdisjoint(groups[confirmation])\\n    dev_features={name:x[dev] for name,x in train_features.items()}\\n    comparison=nested_compare(dev_features,y[dev],groups[dev],site_folds(groups[dev],3,seed=2026),inner_splits=3)\\n    scores=comparison[\'scores\']\\n    plan=selection_plan(scores)\\n    candidate=plan[\'candidate\']\\n    # Freeze the feature family before examining confirmation labels.\\n    selection={**plan,\'basis\':\'development-only control and single challenger selection; simpler families win numerical ties\',\\n               \'development_sites\':len(set(groups[dev])),\'confirmation_sites\':len(set(groups[confirmation])),\\n               \'development_comparison\':_public_summary(comparison)}\\n    write_json(output/\'selection_before_confirmation.json\',selection)\\n    print(\'Feature family frozen before confirmation:\',candidate,flush=True)\\n    confirmation_scores={}\\n    for name in dict.fromkeys([\'table\',\'official\',candidate]):\\n        pred,_=final_predict({k:v[dev] for k,v in train_features.items()},\\n                             {k:v[confirmation] for k,v in train_features.items()},y[dev],groups[dev],name,n_splits=3)\\n        confirmation_scores[name]=metric(y[confirmation],pred)\\n        print(\'Confirmation\',name,confirmation_scores[name][\'mcrmse\'],flush=True)\\n    # Confirmation is an accept/reject gate, so its selected result is not an untouched test estimate.\\n    retained,gate=confirm_selection(plan,confirmation_scores)\\n    prediction,choices=final_predict(train_features,test_features,y,groups,retained,n_splits=5)\\n    export_hash=export_submission(sample,prediction,output/\'submission_selected.csv\')\\n    official_prediction,official_choices=final_predict(train_features,test_features,y,groups,\'official\',n_splits=5)\\n    official_hash=export_submission(sample,official_prediction,output/\'submission_official_control.csv\')\\n    receipt={\'schema\':2,\'experiment\':\'cloud_spatial_nested_site_selection\',\\n             \'si_disclosure\':\'OpenAI Codex assisted code, tests, analysis implementation and documentation. Competition observations retain original provenance.\',\\n             \'counts\':{\'train\':len(train),\'test\':len(test),\'sites\':len(set(groups))},\\n             \'feature_extraction\':feature_receipt,\'selection\':selection,\'confirmation\':confirmation_scores,\\n             \'retained_variant\':retained,\'confirmation_gate\':gate,\'retained_choices\':choices,\'official_control_choices\':official_choices,\\n             \'parameters\':PARAMS,\'final_fit\':\'all permitted training rows with observed target masks; final weights from whole-site OOF\',\\n             \'environment\':{\'python\':platform.python_version(),**{p:importlib.metadata.version(p) for p in [\'numpy\',\'pandas\',\'scikit-learn\',\'rasterio\',\'scipy\']}},\\n             \'submissions\':{\'selected\':{\'rows\':len(test),\'sha256\':export_hash,\'file\':\'submission_selected.csv\'},\\n                            \'official_control\':{\'rows\':len(test),\'sha256\':official_hash,\'file\':\'submission_official_control.csv\'}},\\n             \'seconds\':round(time.monotonic()-started,3),\\n             \'limitations\':[\'Confirmation is used for an accept/reject gate; its selected result is not an untouched estimate.\',\\n                            \'Gate v2 was fixed during initial image extraction, before any image development/confirmation scores were observed; it also requires novel changes to beat the official control.\',\\n                            \'The earlier raw baseline aggregate evaluated all training sites before this refinement, including confirmation sites.\',\\n                            \'No true forward-time validation is possible with withheld dates. Private-test error remains unknown.\',\\n                            \'Feature-family selection uses development outer scores; those selected scores have model-selection optimism.\']}\\n    write_json(output/\'experiment_receipt.json\',receipt)\\n    print(\'EXPERIMENT_COMPLETE\',flush=True)\\n    print({\'development\':{name:value[\'mcrmse\'] for name,value in scores.items()},\'candidate\':candidate,\'retained\':retained,\\n           \'confirmation\':{name:value[\'mcrmse\'] for name,value in confirmation_scores.items()},\'seconds\':receipt[\'seconds\']},flush=True)\\n    return receipt\\n", "features.py": "\\"\\"\\"Content-only chip summaries; explicit cloud ablations, no geospatial metadata.\\n\\nThe official feature recipe is adapted from Zelzy\'s Apache-2.0 starter.\\nOpenAI Codex assisted the implementation and tests.\\n\\"\\"\\"\\nimport numpy as np\\n\\nBAND_ORDER = [\'B1\',\'B2\',\'B3\',\'B4\',\'B5\',\'B6\',\'B7\',\'B8\',\'B8A\',\'B9\',\'B11\',\'B12\',\'SCL\',\'CLDPRB\']\\nSTAT_BANDS = [\'B2\',\'B3\',\'B4\',\'B8\',\'B11\',\'B12\']\\nSCL_CLASSES = [2,3,4,5,6,7,8,9,10,11]\\nMIN_CLEAR = 500\\n\\n\\ndef _statistics(values, prefix):\\n    if not values.size:\\n        return {f\'{prefix}_{name}\': float(\'nan\') for name in [\'mean\',\'std\',\'p50\',\'p95\',\'p99\',\'max\']}\\n    p50,p95,p99 = np.percentile(values,[50,95,99])\\n    return {f\'{prefix}_{name}\':float(value) for name,value in\\n            zip([\'mean\',\'std\',\'p50\',\'p95\',\'p99\',\'max\'],[values.mean(),values.std(),p50,p95,p99,values.max()])}\\n\\n\\ndef _spatial(bands, mask, sufficient):\\n    names = [f\'s_{band}_{stat}\' for band in [\'B11\',\'B12\'] for stat in [\'centre_mean\',\'outer_mean\',\'centre_minus_outer\']]\\n    names += [\'s_ndvi_centre_minus_outer\',\'s_bsi_centre_minus_outer\',\'s_B12_tail_contrast\',\'s_B11_tail_contrast\',\\n              \'s_B12_centre_std\',\'s_B12_gradient\',\'s_B11_gradient\',\'s_ndvi_gradient\',\'s_centre_clear_fraction\',\'s_outer_clear_fraction\']\\n    if not sufficient:\\n        return dict.fromkeys(names,float(\'nan\'))\\n    height,width = mask.shape\\n    rows,cols = np.indices(mask.shape)\\n    centre = (rows>=height/4)&(rows<3*height/4)&(cols>=width/4)&(cols<3*width/4)\\n    c,edge = mask&centre,mask&~centre\\n    output = {}\\n    for band in [\'B11\',\'B12\']:\\n        a = bands[band]\\n        cm = float(a[c].mean()) if c.sum()>=20 else float(\'nan\')\\n        em = float(a[edge].mean()) if edge.sum()>=20 else float(\'nan\')\\n        output.update({f\'s_{band}_centre_mean\':cm,f\'s_{band}_outer_mean\':em,f\'s_{band}_centre_minus_outer\':cm-em})\\n    ndvi = (bands[\'B8\']-bands[\'B4\'])/(bands[\'B8\']+bands[\'B4\']+1e-6)\\n    bsi = ((bands[\'B11\']+bands[\'B4\'])-(bands[\'B8\']+bands[\'B2\']))/(bands[\'B11\']+bands[\'B4\']+bands[\'B8\']+bands[\'B2\']+1e-6)\\n    for label,a in [(\'ndvi\',ndvi),(\'bsi\',bsi)]:\\n        output[f\'s_{label}_centre_minus_outer\'] = float(a[c].mean()-a[edge].mean()) if min(c.sum(),edge.sum())>=20 else float(\'nan\')\\n    for band in [\'B12\',\'B11\']:\\n        p50,p99 = np.percentile(bands[band][mask],[50,99])\\n        output[f\'s_{band}_tail_contrast\'] = float(p99-p50)\\n    output[\'s_B12_centre_std\'] = float(bands[\'B12\'][c].std()) if c.sum()>=20 else float(\'nan\')\\n    for label,a in [(\'B12\',bands[\'B12\']),(\'B11\',bands[\'B11\']),(\'ndvi\',ndvi)]:\\n        vertical = np.abs(np.diff(a,axis=0))[mask[1:]&mask[:-1]]\\n        horizontal = np.abs(np.diff(a,axis=1))[mask[:,1:]&mask[:,:-1]]\\n        values = np.concatenate([vertical,horizontal])\\n        output[f\'s_{label}_gradient\'] = float(values.mean()) if values.size>=20 else float(\'nan\')\\n    output[\'s_centre_clear_fraction\'] = float(mask[centre].mean())\\n    output[\'s_outer_clear_fraction\'] = float(mask[~centre].mean())\\n    return output\\n\\n\\ndef chip_summary(chip, method=\'official\', spatial=False):\\n    chip = np.asarray(chip)\\n    if chip.ndim!=3 or chip.shape[0]!=14 or min(chip.shape[1:])<8:\\n        raise ValueError(\'Expected fourteen chip layers and at least eight pixels per dimension\')\\n    if method not in {\'official\',\'no_fallback\',\'strict\'}:\\n        raise ValueError(\'Unknown cloud-mask method\')\\n    if not np.isfinite(chip).all():\\n        raise ValueError(\'Unexpected nonfinite source chip\')\\n    scl,cloud = chip[12],chip[13]\\n    has_data = chip[3]>0\\n    clear = has_data&~np.isin(scl,[3,8,9,10])\\n    if method==\'strict\': clear = has_data&np.isin(scl,[4,5,6])&(cloud<20)\\n    sufficient = clear.sum()>=MIN_CLEAR\\n    mask = has_data if method==\'official\' and not sufficient else clear\\n    bands = {name:chip[index].astype(np.float64)*1e-4 for index,name in enumerate(BAND_ORDER[:12])}\\n    available = bool(mask.any()) if method==\'official\' else bool(sufficient)\\n    output = {}\\n    for band in STAT_BANDS:\\n        values = bands[band][mask] if available else np.array([])\\n        if method==\'official\' and not values.size: values=np.zeros(1)\\n        output.update(_statistics(values,f\'c_{band}\'))\\n    if available:\\n        b4,b8,b11,b12 = (bands[name][mask] for name in [\'B4\',\'B8\',\'B11\',\'B12\'])\\n        ndvi = (b8-b4)/(b8+b4+1e-6)\\n        values = [ndvi.std(),np.percentile(ndvi,95),np.percentile(b12/(b11+1e-6),95),\\n                  (b12>.2).mean(),(b12>.4).mean(),(b11>.4).mean()]\\n    else:\\n        values = [0.]*6 if method==\'official\' else [float(\'nan\')]*6\\n    output.update(dict(zip([\'c_ndvi_std\',\'c_ndvi_p95\',\'c_b12_b11_p95\',\'c_hot12_gt_0.2\',\'c_hot12_gt_0.4\',\'c_hot11_gt_0.4\'],map(float,values))))\\n    for cls in SCL_CLASSES: output[f\'c_scl_{cls}\']=float((scl==cls).mean())\\n    output[\'c_cloudprob\']=float(cloud.mean())\\n    output[\'c_valid_frac\']=float(mask.mean())\\n    if method!=\'official\':\\n        output[\'q_insufficient_clear\']=float(not sufficient)\\n        output[\'q_clear_count\']=float(clear.sum())\\n        output[\'q_cloud_high_fraction\']=float((cloud>=40).mean())\\n    if spatial: output.update(_spatial(bands,mask,available))\\n    return output\\n", "selection.py": "\\"\\"\\"Target-wise conservative mixtures with selection confined to inner sites.\\"\\"\\"\\nimport time\\nimport numpy as np\\nfrom .core import TARGETS, metric, site_folds\\nfrom .baseline import fit_predict\\n\\nAMPLITUDES = [0.,.25,.5,.75,1.]\\nIMAGE_WEIGHTS = [0.,.5,1.]\\n\\n\\ndef mix_predictions(base, enhanced, mean, amplitude, image_weight):\\n    if not (0<=amplitude<=1 and 0<=image_weight<=1):\\n        raise ValueError(\'Only convex prediction mixtures are permitted\')\\n    base,enhanced,mean = map(np.asarray,[base,enhanced,mean])\\n    return mean+amplitude*((1-image_weight)*(base-mean)+image_weight*(enhanced-mean))\\n\\n\\ndef choose_mix(truth, base, enhanced, means, image_weights=None):\\n    y,b,e,m = map(lambda a:np.asarray(a,dtype=float),[truth,base,enhanced,means])\\n    if any(a.shape!=y.shape for a in [b,e,m]) or not all(np.isfinite(a).all() for a in [b,e,m]):\\n        raise ValueError(\'Mixture inputs must have matching shapes and finite predictions\')\\n    keep = np.isfinite(y)\\n    if np.isinf(y).any() or keep.sum()<2:\\n        raise ValueError(\'Need at least two measured, finite tuning labels\')\\n    best=None\\n    for w in (IMAGE_WEIGHTS if image_weights is None else image_weights):\\n        for a in AMPLITUDES:\\n            p=mix_predictions(b,e,m,a,w)\\n            error=float(np.sqrt(np.mean((y[keep]-p[keep])**2)))\\n            key=(error,w,a)\\n            if best is None or error<best[0][0]-1e-12 or (abs(error-best[0][0])<=1e-12 and (w,a)<best[0][1:]):\\n                best=(key,{\'amplitude\':a,\'image_weight\':w,\'inner_rmse\':error})\\n    return best[1]\\n\\n\\ndef raw_oof(x,y,groups,n_splits=3,params=None,fitter=None):\\n    fitter=fit_predict if fitter is None else fitter\\n    y=np.asarray(y,dtype=float)\\n    predictions=np.full(y.shape,np.nan)\\n    means=np.full(y.shape,np.nan)\\n    for fit,check in site_folds(groups,n_splits):\\n        for j in range(5):\\n            keep=np.isfinite(y[fit,j])\\n            if len(set(np.asarray(groups)[fit][keep]))<2:\\n                raise ValueError(\'Need at least two label-bearing fitting sites per target\')\\n            _,predictions[check,j]=fitter(x[fit],y[fit,j],x[check],params)\\n            means[check,j]=np.nanmean(y[fit,j])\\n    if not np.isfinite(predictions).all() or not np.isfinite(means).all():\\n        raise AssertionError(\'Incomplete inner cross-validation\')\\n    return predictions,means\\n\\n\\ndef nested_compare(features,truth,groups,folds=None,inner_splits=3,params=None,fitters=None):\\n    y=np.asarray(truth,dtype=float)\\n    groups=np.asarray(groups)\\n    features={key:np.asarray(value,dtype=float) for key,value in features.items()}\\n    if \'table\' not in features or any(len(value)!=len(y) for value in features.values()):\\n        raise ValueError(\'Features require the shared table reference and matching rows\')\\n    folds=site_folds(groups,3) if folds is None else folds\\n    predictions={key:np.full(y.shape,np.nan) for key in features}\\n    raw={key:np.full(y.shape,np.nan) for key in features}\\n    means=np.full(y.shape,np.nan)\\n    reports=[]\\n    fitters={} if fitters is None else fitters\\n    seen=np.zeros(len(y),dtype=int)\\n    for number,(fit,check) in enumerate(folds,1):\\n        tick=time.monotonic()\\n        if set(groups[fit])&set(groups[check]) or np.intersect1d(fit,check).size:\\n            raise ValueError(\'Outer facility leakage\')\\n        seen[check]+=1\\n        inner={}\\n        for name,x in features.items():\\n            inner[name],inner_means=raw_oof(x[fit],y[fit],groups[fit],inner_splits,params,fitters.get(name))\\n        for name,x in features.items():\\n            for j in range(5):\\n                _,raw[name][check,j]=fitters.get(name,fit_predict)(x[fit],y[fit,j],x[check],params)\\n        means[check]=np.nanmean(y[fit],axis=0)\\n        choices={}\\n        for name in features:\\n            choices[name]=[]\\n            for j in range(5):\\n                choice=choose_mix(y[fit,j],inner[\'table\'][:,j],inner[name][:,j],inner_means[:,j],\\n                                  image_weights=[0.] if name==\'table\' else None)\\n                choices[name].append(choice)\\n                predictions[name][check,j]=mix_predictions(raw[\'table\'][check,j],raw[name][check,j],\\n                    means[check,j],choice[\'amplitude\'],choice[\'image_weight\'])\\n        reports.append({\'fold\':number,\'fit_sites\':len(set(groups[fit])),\'check_sites\':len(set(groups[check])),\\n                        \'choices\':choices,\'scores\':{name:metric(y[check],p[check]) for name,p in predictions.items()},\\n                        \'seconds\':round(time.monotonic()-tick,3)})\\n        print(\'Outer fold\',number,\'scores:\',{name:round(value[\'mcrmse\'],6) for name,value in reports[-1][\'scores\'].items()},flush=True)\\n    if not np.all(seen==1):\\n        raise ValueError(\'Outer folds must hold every row out exactly once\')\\n    scores={name:metric(y,p) for name,p in predictions.items()}\\n    return {\'scores\':scores,\'folds\':reports,\'predictions\':predictions,\'raw_predictions\':raw,\\n            \'mean_predictions\':means,\'fold_mean_score\':metric(y,means),\\n            \'raw_scores\':{name:metric(y,p) for name,p in raw.items()}}\\n\\n\\ndef final_predict(train_features,test_features,truth,groups,variant,n_splits=5,params=None,fitters=None):\\n    y=np.asarray(truth,dtype=float)\\n    names=[\'table\'] if variant==\'table\' else [\'table\',variant]\\n    oof={}\\n    test_predictions={}\\n    fitters={} if fitters is None else fitters\\n    for name in names:\\n        oof[name],means=raw_oof(np.asarray(train_features[name]),y,groups,n_splits,params,fitters.get(name))\\n        test_predictions[name]=np.zeros((len(test_features[name]),5))\\n        for j in range(5):\\n            _,test_predictions[name][:,j]=fitters.get(name,fit_predict)(train_features[name],y[:,j],test_features[name],params)\\n    mu=np.nanmean(y,axis=0)\\n    result=np.zeros_like(test_predictions[\'table\'])\\n    choices=[]\\n    for j in range(5):\\n        choice=choose_mix(y[:,j],oof[\'table\'][:,j],oof[variant][:,j],means[:,j],\\n                          image_weights=[0.] if variant==\'table\' else None)\\n        result[:,j]=mix_predictions(test_predictions[\'table\'][:,j],test_predictions[variant][:,j],\\n                                    mu[j],choice[\'amplitude\'],choice[\'image_weight\'])\\n        choices.append(choice)\\n    if not np.isfinite(result).all(): raise ValueError(\'Nonfinite final prediction\')\\n    return result,choices\\n"}')
for name, text in sources.items():
    (package/name).write_text(text, encoding='utf-8')
for name in list(sys.modules):
    if name == 'atmospheric_lab' or name.startswith('atmospheric_lab.'):
        del sys.modules[name]
print('LavaLabs source ready; aggregate reporting only.')


In [ ]:
RUN_CAT_IN_PRIVATE_COPY = False
if not RUN_CAT_IN_PRIVATE_COPY:
    print('Source-only CatBoost release. Reproduce only in a private copy; keep row predictions private.')
else:
    from pathlib import Path
    from atmospheric_lab.cat_followup import run
    roots = [Path('/kaggle/input/competitions/obscura'),Path('/kaggle/input/obscura')]
    data = next((p for p in roots if (p/'train.csv').exists()),None)
    if data is None:
        raise RuntimeError('Attach the official OBSCURA competition input before running.')
    receipt = run(data, '/kaggle/working/lavalabs_private_catboost')
    print('Private CSV ready. Do not publish row-level outputs.')
    print('AGGREGATE_RECEIPT_BEGIN')
    print(json.dumps(receipt, sort_keys=True))
    print('AGGREGATE_RECEIPT_END')


## Sparse spectral FIRMS follow-up, 23 November 2026
Independent implementation motivated by the fire/SWIR literature. Inputs remain only the official 29 table features and supplied L2A chips. No outside observations, recovered geolocation, pretrained weights, physical fire retrieval or calibrated emissions. The 99-feature branch adds four quality and twelve sparse spectral/context summaries to the official 83-feature recipe. Only FIRMS changes; four gas/aerosol outputs remain exactly equal to the frozen control.
Fixed controls: quality-only, brightness-only, six-descriptor, full twelve-descriptor and spatially shuffled. The full branch is the sole predeclared promotion candidate. All mixtures are chosen on inner sites. On the same 64 development sites/2,597 rows: control MCRMSE 0.9337828783; full spectral 0.9194996153; quality-only 0.9337952465; brightness-only 0.9306248678; shuffled 0.9198552401. Full candidate wins 2/3 outer folds. FIRMS RMSE gain 0.0714163; all other target errors unchanged.
Two feature-only environmental cluster holdouts: pooled control 0.9662826536 versus full spectral 0.9462006522. Conditional fixed-OOF paired-site bootstrap 95 percentile gain interval [0.00228788, 0.02501520]; excludes retraining and selection uncertainty. Reused development is not independent confirmation, a true forward-time test or private leaderboard accuracy. The shuffled control is almost as strong, so spatial structure alone is not established as the cause of improvement.
Nine feature/controller checks and two refit safety checks passed. The accepted full fit reproduced the old complete CSV SHA-256 exactly before replacing only FIRMS. Candidate private CSV SHA-256: ae6287bd5aee9269d6d5eb6ffe8e77f7b60db3714c7b275e59154474a2a2055a. No prediction rows are included here. Check the release receipt for fresh saved replay and genuine public score; development is not the public score.
Methods: Massimetti et al. 2020 https://www.mdpi.com/2072-4292/12/5/820 ; Fisher and Wooster 2018 https://www.mdpi.com/2072-4292/10/2/305 ; Marchese et al. 2019 https://www.mdpi.com/2072-4292/11/23/2876 . Their physical retrievals/thresholds are not claimed to transfer directly to these L2A inputs. No third-party fire implementation is vendored. SI-assisted implementation/tests/analysis; source observations unchanged.
To reproduce: make a PRIVATE copy, attach official OBSCURA data, CPU two threads, and enable only RUN_SPECTRAL_IN_PRIVATE_COPY. Keep outputs/chips/IDs/feature matrices/OOF arrays/CSVs private. First inspect the label-blind profile; this all-in-one reproduction uses fixed thresholds and does not retune using that inspection.


In [ ]:
RUN_SPECTRAL_IN_PRIVATE_COPY = False
if not RUN_SPECTRAL_IN_PRIVATE_COPY:
    print('Source-only spectral release. Reproduce in a private copy; keep competition outputs private.')
else:
    # Private scientific experiment. No downloads or external observations.
    import os
    for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']:
        os.environ[key]='2'
    from pathlib import Path
    import json,hashlib,sys,subprocess
    import pandas as pd
    sources=json.loads('{"spectral_lab/__init__.py": "\\"\\"\\"Isolated L2A sparse spectral experiment. SI-assisted source; no physical retrieval.\\"\\"\\"\\n", "spectral_lab/audit_saved.py": "\\"\\"\\"Separate disk-to-truth verification of saved private spectral outputs.\\"\\"\\"\\nfrom pathlib import Path\\nimport hashlib,json\\nimport numpy as np\\nfrom sklearn.model_selection import GroupShuffleSplit\\nfrom .core import TARGETS,load_tables,write_json\\nfrom .independent_audit import independent_score,paired_site_summary\\n\\ndef audit(root,output):\\n    output=Path(output);receipt=json.loads((output/\'spectral_receipt.json\').read_text())\\n    train,_,_,names=load_tables(root);groups=train.site_key.to_numpy()\\n    dev,_=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=2026).split(train[names],groups=groups))\\n    truth=train[TARGETS].to_numpy(dtype=float)[dev];groups=groups[dev].astype(str)\\n    result={\'schema\':1,\'outputs\':{},\'rows_exported\':False,\'test_labels_accessed\':False}\\n    for name,file,key in [(\'development\',\'private_development_predictions.npz\',\'development\'),(\'environment\',\'private_environment_predictions.npz\',\'environment_stress\')]:\\n        path=output/file\\n        with np.load(path,allow_pickle=False) as saved:\\n            if not np.array_equal(truth,saved[\'truth\'],equal_nan=True) or not np.array_equal(groups,saved[\'groups\']):raise ValueError(\'Saved truth/group mismatch\')\\n            scores={variant:independent_score(truth,saved[variant]) for variant in [\'C\',\'Q\',\'B\',\'A\',\'F\',\'S\']}\\n            for variant,score in scores.items():\\n                stated=receipt[key][\'scores\'][variant]\\n                if abs(score[\'mcrmse\']-stated[\'mcrmse\'])>1e-12 or score[\'observed\']!=stated[\'observed\']:raise ValueError(\'Saved metric mismatch\')\\n                for t in TARGETS:\\n                    if abs(score[\'rmse\'][t]-stated[\'rmse\'][t])>1e-12:raise ValueError(\'Target RMSE mismatch\')\\n                if not np.array_equal(saved[\'C\'][:,:4],saved[variant][:,:4]):raise ValueError(\'Protected target mismatch\')\\n            if name==\'development\':\\n                boot=paired_site_summary(truth,saved[\'C\'],saved[\'F\'],groups)\\n                if boot!=receipt[\'paired_sites\']:raise ValueError(\'Bootstrap replay mismatch\')\\n        result[\'outputs\'][name]={\'scores\':scores,\'sha256\':hashlib.sha256(path.read_bytes()).hexdigest(),\'truth_and_sites_verified\':True}\\n    result[\'passed\']=True\\n    write_json(output/\'independent_spectral_audit.json\',result)\\n    print(\'SPECTRAL_AUDIT_BEGIN\');print(json.dumps(result,sort_keys=True));print(\'SPECTRAL_AUDIT_END\',flush=True)\\n    return result\\n", "spectral_lab/baseline.py": "\\"\\"\\"First measured baseline. Official HistGB recipe, supplied features, no selection.\\"\\"\\"\\nfrom pathlib import Path\\nimport importlib.metadata\\nimport platform\\nimport time\\nimport numpy as np\\nfrom sklearn.ensemble import HistGradientBoostingRegressor\\nfrom threadpoolctl import threadpool_limits\\nfrom .core import TARGETS, SOURCE, load_tables, site_folds, metric, export_submission, write_json\\n\\nPARAMS = dict(max_iter=250, learning_rate=.04, max_depth=3,\\n              min_samples_leaf=40, l2_regularization=5., max_features=.6,\\n              random_state=0, early_stopping=False)\\n\\n\\ndef fit_predict(x, y, z, params=None):\\n    y = np.asarray(y, dtype=float)\\n    keep = np.isfinite(y)\\n    if keep.sum() < 2:\\n        raise ValueError(\'Not enough measured labels in the fitting partition\')\\n    with threadpool_limits(limits=2):\\n        model = HistGradientBoostingRegressor(**(PARAMS if params is None else params))\\n        model.fit(np.asarray(x)[keep], y[keep])\\n        prediction = model.predict(z)\\n    if not np.isfinite(prediction).all():\\n        raise ValueError(\'Estimator produced a nonfinite prediction\')\\n    return model, prediction\\n\\n\\ndef run(root, output, n_splits=5, params=None, expected_rows=1221):\\n    started = time.monotonic()\\n    output = Path(output)\\n    output.mkdir(parents=True, exist_ok=True)\\n    train, test, sample, names = load_tables(root, expected_rows)\\n    x = train[names].to_numpy(dtype=float)\\n    z = test[names].to_numpy(dtype=float)\\n    y = train[TARGETS].to_numpy(dtype=float)\\n    folds = site_folds(train.site_key.to_numpy(), n_splits)\\n    oof = np.full(y.shape, np.nan)\\n    mean_oof = np.full(y.shape, np.nan)\\n    fold_reports = []\\n    for fold_number, (fit, check) in enumerate(folds, 1):\\n        tick = time.monotonic()\\n        for j in range(5):\\n            _, oof[check, j] = fit_predict(x[fit], y[fit, j], x[check], params)\\n            mean_oof[check, j] = np.nanmean(y[fit, j])\\n        report = metric(y[check], oof[check])\\n        report.update(fold=fold_number, seconds=round(time.monotonic()-tick, 3),\\n                      heldout_sites=int(train.iloc[check].site_key.nunique()),\\n                      fit_sites=int(train.iloc[fit].site_key.nunique()))\\n        fold_reports.append(report)\\n        print(f\'Fold {fold_number}/{n_splits}: MCRMSE {report[\\"mcrmse\\"]:.6f}; {report[\\"seconds\\"]:.1f}s\', flush=True)\\n    predictions = np.zeros((len(test), 5))\\n    for j in range(5):\\n        _, predictions[:, j] = fit_predict(x, y[:, j], z, params)\\n    export_hash = export_submission(sample, predictions, output/\'submission_table.csv\')\\n    receipt = {\'schema\': 1, \'experiment\': \'supplied_table_unselected_histgb\',\\n               \'si_disclosure\': \'OpenAI Codex assisted code, tests and documentation; source observations retain competition provenance.\',\\n               \'baseline_reference\': SOURCE, \'baseline_adaptation\': \'29 supplied features; explicit no early stopping; no OOF-tuned shrinkage.\',\\n               \'counts\': {\'train\': len(train), \'test\': len(test), \'sites\': int(train.site_key.nunique()), \'features\': len(names)},\\n               \'validation\': {\'method\': \'whole-site GroupKFold\', \'folds\': n_splits, \'site_overlap\': False,\\n                              \'selection\': \'none; fixed baseline parameters\', \'pooled_model\': metric(y, oof),\\n                              \'pooled_fold_mean\': metric(y, mean_oof), \'pooled_zero\': metric(y, np.zeros_like(y)),\\n                              \'fold_reports\': fold_reports},\\n               \'parameters\': PARAMS if params is None else params, \'seconds\': round(time.monotonic()-started, 3),\\n               \'environment\': {\'python\': platform.python_version(), **{p: importlib.metadata.version(p) for p in [\'numpy\',\'pandas\',\'scikit-learn\',\'scipy\']}},\\n               \'submission\': {\'rows\': len(test), \'sha256\': export_hash, \'finite\': True, \'private_file\': \'submission_table.csv\'},\\n               \'boundary\': \'Unseen-site validation within training time; private/future-time performance is unknown. No leaderboard score yet.\'}\\n    write_json(output/\'baseline_receipt.json\', receipt)\\n    print(\'BASELINE_COMPLETE\', flush=True)\\n    print(receipt[\'validation\'][\'pooled_model\'], flush=True)\\n    return receipt\\n", "spectral_lab/comparison.py": "\\"\\"\\"Frozen grouped evaluation. Only FIRMS may change; no outer-label selection.\\"\\"\\"\\nfrom pathlib import Path\\nimport hashlib,json,time,platform,importlib.metadata\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.cluster import KMeans\\nfrom sklearn.model_selection import GroupShuffleSplit\\nfrom .core import TARGETS,load_tables,metric,site_folds,write_json\\nfrom .baseline import fit_predict\\nfrom .selection import choose_mix,mix_predictions\\nfrom .independent_audit import independent_score,paired_site_summary\\nfrom .extractor import matrices\\n\\nVARIANTS=(\'Q\',\'B\',\'A\',\'F\',\'S\')\\nHISTORICAL_CONTROL=.9337828782984863\\n\\ndef branch_mix(b,e,f,m,a,w,v):\\n    return m+a*((1-v)*((1-w)*(b-m)+w*(e-m))+v*(f-m))\\n\\ndef choose_branch(y,b,e,f,m):\\n    arrays=[np.asarray(x,dtype=float) for x in [y,b,e,f,m]]\\n    y,b,e,f,m=arrays;keep=np.isfinite(y)\\n    if keep.sum()<2 or any(x.shape!=y.shape for x in arrays) or np.isinf(y).any() or not all(np.isfinite(x).all() for x in arrays[1:]):\\n        raise ValueError(\'Invalid branch selection inputs\')\\n    best=None\\n    for v in [0.,.5,1.]:\\n        for w in [0.,.5,1.]:\\n            for a in [0.,.25,.5,.75,1.]:\\n                p=branch_mix(b,e,f,m,a,w,v)\\n                error=float(np.sqrt(np.mean((y[keep]-p[keep])**2)))\\n                key=(error,v,w,a)\\n                if best is None or error<best[0][0]-1e-12 or (abs(error-best[0][0])<=1e-12 and key[1:]<best[0][1:]):\\n                    best=(key,dict(amplitude=a,image_weight=w,branch_weight=v,inner_rmse=error))\\n    return best[1]\\n\\ndef branch_predict(choice,b,e,f,m):\\n    return branch_mix(b,e,f,m,choice[\'amplitude\'],choice[\'image_weight\'],choice[\'branch_weight\'])\\n\\ndef safe_fit(x,y,z):\\n    # Fit-only all-missing columns cannot convey information and can trigger a\\n    # sklearn histogram bug. Never inspect validation rows to decide columns.\\n    measured=np.isfinite(y);columns=np.isfinite(x[measured]).any(axis=0)\\n    if not columns.any():raise ValueError(\'No observed fitting features\')\\n    return fit_predict(x[:,columns],y,z[:,columns])[1]\\n\\ndef outer(features,y,groups,fit,check,fit_fn=safe_fit):\\n    groups=np.asarray(groups);y=np.asarray(y,dtype=float)\\n    if set(groups[fit])&set(groups[check]) or np.intersect1d(fit,check).size:raise ValueError(\'Outer site overlap\')\\n    inner={k:np.full((len(fit),5),np.nan) for k in [\'table\',\'official\',\'mean\']}\\n    extra={k:np.full(len(fit),np.nan) for k in VARIANTS}\\n    for a,b in site_folds(groups[fit],3):\\n        a_global,b_global=fit[a],fit[b]\\n        for j in range(5):\\n            if len(set(groups[a_global][np.isfinite(y[a_global,j])]))<2:raise ValueError(\'Too few observed fitting sites\')\\n            for name in [\'table\',\'official\']:\\n                inner[name][b,j]=fit_fn(features[name][a_global],y[a_global,j],features[name][b_global])\\n        inner[\'mean\'][b]=np.nanmean(y[a_global],axis=0)\\n        for name in VARIANTS:extra[name][b]=fit_fn(features[name][a_global],y[a_global,4],features[name][b_global])\\n    if not all(np.isfinite(a).all() for a in [*inner.values(),*extra.values()]):raise ValueError(\'Missing inner predictions\')\\n    control_choices=[choose_mix(y[fit,j],inner[\'table\'][:,j],inner[\'official\'][:,j],inner[\'mean\'][:,j]) for j in range(5)]\\n    choices={name:choose_branch(y[fit,4],inner[\'table\'][:,4],inner[\'official\'][:,4],extra[name],inner[\'mean\'][:,4]) for name in VARIANTS}\\n    raw={k:np.column_stack([fit_fn(features[k][fit],y[fit,j],features[k][check]) for j in range(5)]) for k in [\'table\',\'official\']}\\n    raw.update({name:fit_fn(features[name][fit],y[fit,4],features[name][check]) for name in VARIANTS})\\n    mu=np.nanmean(y[fit],axis=0)\\n    control=np.column_stack([mix_predictions(raw[\'table\'][:,j],raw[\'official\'][:,j],mu[j],c[\'amplitude\'],c[\'image_weight\']) for j,c in enumerate(control_choices)])\\n    prediction={\'C\':control}\\n    for name in VARIANTS:\\n        prediction[name]=control.copy()\\n        prediction[name][:,4]=branch_predict(choices[name],raw[\'table\'][:,4],raw[\'official\'][:,4],raw[name],mu[4])\\n        if not np.array_equal(prediction[name][:,:4],control[:,:4]):raise AssertionError(\'Protected targets changed\')\\n    return prediction,{\'C\':control_choices,**choices},raw\\n\\ndef evaluate(features,y,groups,folds,label,started,fit_fn=safe_fit):\\n    oof={name:np.full(y.shape,np.nan) for name in [\'C\',*VARIANTS]};reports=[];seen=np.zeros(len(y),int)\\n    for number,(fit,check) in enumerate(folds,1):\\n        if time.monotonic()-started>7200:raise RuntimeError(\'Comparison exceeded fixed two-hour cap\')\\n        tick=time.monotonic();prediction,choices,raw=outer(features,y,groups,fit,check,fit_fn)\\n        seen[check]+=1\\n        for name,p in prediction.items():oof[name][check]=p\\n        scores={name:metric(y[check],p) for name,p in prediction.items()}\\n        report=dict(fold=number,fit_sites=len(set(groups[fit])),check_sites=len(set(groups[check])),scores=scores,choices=choices,\\n                    raw_firms_rmse={name:float(np.sqrt(np.nanmean((y[check,4]-p)**2))) for name,p in raw.items() if name in VARIANTS},seconds=time.monotonic()-tick)\\n        reports.append(report)\\n        print(\'SPECTRAL_FOLD\',label,number,json.dumps(report,allow_nan=False),flush=True)\\n    if not np.all(seen==1):raise ValueError(\'OOF coverage must be exactly once\')\\n    scores={name:metric(y,p) for name,p in oof.items()}\\n    for name,p in oof.items():\\n        audited=independent_score(y,p)\\n        if abs(audited[\'mcrmse\']-scores[name][\'mcrmse\'])>1e-12 or audited[\'observed\']!=scores[name][\'observed\']:raise AssertionError(\'Independent metric mismatch\')\\n    return dict(scores=scores,folds=reports,independently_recomputed=True),oof\\n\\ndef environment_folds(table,groups):\\n    columns=[\'temperature\',\'humidity\',\'wind_speed\',\'precipitation\',\'solar_rad\',\'elevation\',\'slope\']\\n    if not set(columns)<=set(table.columns):raise ValueError(\'Environmental feature contract changed\')\\n    data=table[columns].copy();data[\'group\']=groups\\n    med=data.groupby(\'group\',sort=True)[columns].median()\\n    a=med.to_numpy(dtype=float);fill=np.nanmedian(a,axis=0)\\n    if not np.isfinite(fill).all():raise ValueError(\'Unavailable environmental partition feature\')\\n    a=np.where(np.isfinite(a),a,fill);scale=a.std(axis=0);scale[scale==0]=1\\n    clusters=KMeans(n_clusters=2,random_state=2026,n_init=10).fit_predict((a-a.mean(axis=0))/scale)\\n    if min(np.bincount(clusters))<5:raise ValueError(\'Environmental stress cluster too small\')\\n    mapping=dict(zip(med.index,clusters));row_cluster=np.array([mapping[g] for g in groups])\\n    folds=[(np.flatnonzero(row_cluster!=i),np.flatnonzero(row_cluster==i)) for i in range(2)]\\n    return folds,dict(features=columns,site_counts=np.bincount(clusters).tolist(),row_counts=np.bincount(row_cluster).tolist(),uses_targets=False)\\n\\ndef make_features(x,values):\\n    official=np.column_stack([x,values[\'official\']]);q=np.column_stack([official,values[\'quality\']])\\n    return {\'table\':x,\'official\':official,\'Q\':q,\'B\':np.column_stack([q,values[\'brightness\']]),\\n            \'A\':np.column_stack([q,values[\'thermal\'][:,:6]]),\'F\':np.column_stack([q,values[\'thermal\']]),\\n            \'S\':np.column_stack([q,values[\'scrambled\']])}\\n\\ndef run(root,output):\\n    started=time.monotonic();output=Path(output);output.mkdir(parents=True,exist_ok=True)\\n    if not (output/\'profile_receipt.json\').exists() or not json.loads((output/\'profile_receipt.json\').read_text())[\'passed\']:\\n        raise ValueError(\'Passed label-blind profile required\')\\n    train,test,sample,names=load_tables(root);x=train[names].to_numpy(dtype=float);y=train[TARGETS].to_numpy(dtype=float);groups=train.site_key.to_numpy()\\n    values,extraction=matrices(root,[*train.sample_id,*test.sample_id],output/\'private_features.npz\')\\n    dev,_=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=2026).split(x,groups=groups))\\n    dev_values={k:v[:len(x)][dev] for k,v in values.items()}\\n    f=make_features(x[dev],dev_values);dy=y[dev];dg=groups[dev]\\n    stress_splits,stress_partition=environment_folds(train.iloc[dev],dg)\\n    comparison_tick=time.monotonic()\\n    development,oof=evaluate(f,dy,dg,site_folds(dg,3,seed=2026),\'development\',comparison_tick)\\n    np.savez_compressed(output/\'private_development_predictions.npz\',truth=dy,groups=dg.astype(str),**oof)\\n    write_json(output/\'development_receipt.json\',development)\\n    stress,stress_oof=evaluate(f,dy,dg,stress_splits,\'environment\',comparison_tick)\\n    np.savez_compressed(output/\'private_environment_predictions.npz\',truth=dy,groups=dg.astype(str),**stress_oof)\\n    paired=paired_site_summary(dy,oof[\'C\'],oof[\'F\'],dg)\\n    scores=development[\'scores\'];gain=scores[\'C\'][\'mcrmse\']-scores[\'F\'][\'mcrmse\']\\n    wins=sum(r[\'scores\'][\'F\'][\'mcrmse\']<r[\'scores\'][\'C\'][\'mcrmse\']-1e-10 for r in development[\'folds\'])\\n    target_gain={t:scores[\'C\'][\'rmse\'][t]-scores[\'F\'][\'rmse\'][t] for t in TARGETS}\\n    checks={\'pooled_gain_at_least_005\':gain>=.005,\'at_least_two_fold_wins\':wins>=2,\'no_target_regression_over_02\':min(target_gain.values())>=-.02,\\n            \'beats_quality_control\':scores[\'F\'][\'mcrmse\']<scores[\'Q\'][\'mcrmse\']-1e-10,\'beats_brightness_control\':scores[\'F\'][\'mcrmse\']<scores[\'B\'][\'mcrmse\']-1e-10,\\n            \'no_environment_stress_regression\':stress[\'scores\'][\'F\'][\'mcrmse\']<=stress[\'scores\'][\'C\'][\'mcrmse\']+1e-10,\\n            \'positive_conditional_bootstrap_lower_bound\':paired[\'conditional_site_bootstrap_95_percentile\'][0]>0,\\n            \'historical_control_reconciled\':abs(scores[\'C\'][\'mcrmse\']-HISTORICAL_CONTROL)<=1e-9,\\n            \'protected_targets_byte_equal\':all(np.array_equal(oof[n][:,:4],oof[\'C\'][:,:4]) for n in VARIANTS)}\\n    receipt={\'schema\':1,\'experiment\':\'sparse_local_spectral_firms_v4\',\'development\':development,\'environment_stress\':stress,\\n             \'stress_partition\':stress_partition,\'paired_sites\':paired,\'extraction\':extraction,\\n             \'gate\':dict(passed=all(checks.values()),checks=checks,pooled_gain=gain,fold_wins=wins,per_target_gain=target_gain),\\n             \'counts\':dict(development_rows=len(dev),development_sites=len(set(dg)),train_rows=len(x),test_rows=len(test)),\\n             \'environment\':dict(python=platform.python_version(),**{p:importlib.metadata.version(p) for p in [\'numpy\',\'scipy\',\'scikit-learn\',\'rasterio\']}),\\n             \'seconds\':time.monotonic()-started,\'submission_exported\':False,\\n             \'limits\':[\'Reused development sites; no independent confirmation or time-forward guarantee.\',\\n                       \'Descriptors are L2A spectral shape, not calibrated fire, FRP, temperature or emissions.\',\\n                       \'Only FIRMS changes; first four targets exactly match control.\',\'No scored export before separate fresh-session replay and audit.\'],\\n             \'si_disclosure\':\'OpenAI Codex assisted source and tests; source observations unchanged.\'}\\n    write_json(output/\'spectral_receipt.json\',receipt)\\n    print(\'SPECTRAL_RECEIPT_BEGIN\');print(json.dumps(receipt,sort_keys=True,allow_nan=False));print(\'SPECTRAL_RECEIPT_END\',flush=True)\\n    return receipt\\n", "spectral_lab/core.py": "\\"\\"\\"Contracts and numerical checks for masked, whole-site atmospheric regression.\\"\\"\\"\\nfrom pathlib import Path\\nimport hashlib\\nimport json\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.model_selection import GroupKFold\\n\\nTARGETS = [\'t_ch4\', \'t_no2\', \'t_co\', \'t_aai\', \'t_firms\']\\nSOURCE = \'https://www.kaggle.com/code/zelza0/obscura-starter-baseline\'\\n\\n\\ndef metric(truth, prediction):\\n    \\"\\"\\"Official pooled, masked target RMSEs; never ignore a bad prediction.\\"\\"\\"\\n    y, p = np.asarray(truth, dtype=float), np.asarray(prediction, dtype=float)\\n    if y.ndim != 2 or y.shape != p.shape or y.shape[1] != 5:\\n        raise ValueError(\'Expected equally shaped n-by-five targets and predictions\')\\n    if np.isinf(y).any() or not np.isfinite(p).all():\\n        raise ValueError(\'Targets cannot be infinite; every prediction must be finite\')\\n    present = np.isfinite(y)\\n    counts = present.sum(axis=0)\\n    if (counts == 0).any():\\n        raise ValueError(\'All five targets need at least one observed value to report MCRMSE\')\\n    errors = [float(np.sqrt(np.mean((y[present[:, j], j] - p[present[:, j], j]) ** 2)))\\n              for j in range(5)]\\n    return {\'mcrmse\': float(np.mean(errors)), \'rmse\': dict(zip(TARGETS, errors)),\\n            \'observed\': dict(zip(TARGETS, map(int, counts)))}\\n\\n\\ndef validate_tables(train, test, sample, expected_rows=None):\\n    for name, frame in [(\'train\', train), (\'test\', test), (\'sample\', sample)]:\\n        if frame.columns.duplicated().any() or \'sample_id\' not in frame:\\n            raise ValueError(f\'{name}: duplicate columns or missing identifier\')\\n        ids = frame[\'sample_id\']\\n        if ids.isna().any() or not ids.is_unique or not ids.map(lambda x: isinstance(x, str) and bool(x.strip())).all():\\n            raise ValueError(f\'{name}: identifiers must be unique, nonempty strings\')\\n    if set(train.sample_id) & set(test.sample_id):\\n        raise ValueError(\'Training and test sample IDs overlap\')\\n    if list(sample.columns) != [\'sample_id\', *TARGETS]:\\n        raise ValueError(\'Submission template columns do not match the contract\')\\n    if list(test.sample_id) != list(sample.sample_id):\\n        raise ValueError(\'Test and template identifier order differs\')\\n    if expected_rows is not None and len(test) != expected_rows:\\n        raise ValueError(\'Unexpected test row count\')\\n    if \'site_key\' not in train or train.site_key.isna().any():\\n        raise ValueError(\'Training sites must be present\')\\n    features = [c for c in test if c != \'sample_id\']\\n    if any(c in features for c in [\'site_key\', *TARGETS]):\\n        raise ValueError(\'Label or site identifier found among predictors\')\\n    if set(train.columns) != {\'sample_id\', \'site_key\', *TARGETS, *features}:\\n        raise ValueError(\'Training and test feature schemas differ\')\\n    for frame, names in [(train, [*features, *TARGETS]), (test, features)]:\\n        for name in names:\\n            if not pd.api.types.is_numeric_dtype(frame[name]) or np.isinf(frame[name].to_numpy(dtype=float)).any():\\n                raise ValueError(\'Predictors and targets must be numerical, missing or finite\')\\n    if train[TARGETS].notna().sum().eq(0).any():\\n        raise ValueError(\'A training target has no observations\')\\n    return features\\n\\n\\ndef load_tables(root, expected_rows=1221):\\n    root = Path(root)\\n    train = pd.read_csv(root / \'train.csv\', dtype={\'sample_id\': str, \'site_key\': str})\\n    test = pd.read_csv(root / \'test.csv\', dtype={\'sample_id\': str})\\n    sample = pd.read_csv(root / \'sample_submission.csv\', dtype={\'sample_id\': str})\\n    features = validate_tables(train, test, sample, expected_rows)\\n    if len(features) != 29:\\n        raise ValueError(\'Expected the official 29 supplied numerical features\')\\n    return train, test, sample, features\\n\\n\\ndef site_folds(groups, n_splits=5, seed=None):\\n    groups = np.asarray(groups)\\n    if pd.isna(groups).any() or len(set(groups)) < n_splits:\\n        raise ValueError(\'Not enough valid sites for the requested folds\')\\n    splitter = GroupKFold(n_splits=n_splits) if seed is None else GroupKFold(n_splits=n_splits,shuffle=True,random_state=seed)\\n    folds = list(splitter.split(np.zeros(len(groups)), groups=groups))\\n    seen = np.zeros(len(groups), dtype=int)\\n    for fit, check in folds:\\n        if set(groups[fit]) & set(groups[check]):\\n            raise AssertionError(\'Site leakage across a fold\')\\n        seen[check] += 1\\n    if not np.all(seen == 1):\\n        raise AssertionError(\'Every row must be held out exactly once\')\\n    return folds\\n\\n\\ndef export_submission(sample, predictions, destination):\\n    p = np.asarray(predictions, dtype=float)\\n    if p.shape != (len(sample), 5) or not np.isfinite(p).all():\\n        raise ValueError(\'Test predictions must be finite and match all template rows\')\\n    if not sample.sample_id.is_unique or not sample.sample_id.map(lambda x: isinstance(x, str)).all():\\n        raise ValueError(\'Invalid submission identifiers\')\\n    result = sample[[\'sample_id\']].copy()\\n    result[TARGETS] = p\\n    path = Path(destination)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    result.to_csv(path, index=False, float_format=\'%.9g\')\\n    replay = pd.read_csv(path, dtype={\'sample_id\': str})\\n    if list(replay.sample_id) != list(sample.sample_id) or not np.isfinite(replay[TARGETS]).all().all():\\n        raise AssertionError(\'Export round trip failed\')\\n    return hashlib.sha256(path.read_bytes()).hexdigest()\\n\\n\\ndef write_json(path, value):\\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\\\n\', encoding=\'utf-8\')\\n", "spectral_lab/extractor.py": "\\"\\"\\"Private single-pass official/sparse feature extraction; aggregate receipts only.\\"\\"\\"\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom pathlib import Path\\nimport hashlib,inspect,json,time,warnings\\nimport numpy as np\\nfrom . import hotspots,features\\nfrom .hotspots import THERMAL_NAMES,QUALITY_NAMES,BRIGHTNESS_NAMES\\n\\ndef read_one(root,sid):\\n    import rasterio\\n    from rasterio.errors import NotGeoreferencedWarning\\n    try:\\n        with warnings.catch_warnings():\\n            warnings.simplefilter(\'ignore\',NotGeoreferencedWarning)\\n            with rasterio.open(Path(root)/\'images\'/f\'{sid}.tif\') as source:chip=source.read()\\n        if chip.dtype!=np.uint16 or chip.shape!=(14,384,384):raise ValueError(\'Source array contract mismatch\')\\n        values,diagnostic=hotspots.extract(chip)\\n        official=features.chip_summary(chip,\'official\')\\n        values[\'official\']=np.array([official[n] for n in sorted(official)])\\n        return values,diagnostic\\n    except Exception:\\n        raise RuntimeError(\'A private source chip failed extraction; no fabricated fallback.\') from None\\n\\ndef summarize(diagnostics):\\n    return {name:{\'mean\':float(np.mean([d[name] for d in diagnostics])),\\n                  \'max\':float(np.max([d[name] for d in diagnostics])),\\n                  \'min\':float(np.min([d[name] for d in diagnostics]))} for name in diagnostics[0]}\\n\\ndef profile(root,train_ids,output):\\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\\n    chosen=np.random.default_rng(2026).choice(len(train_ids),size=min(32,len(train_ids)),replace=False)\\n    tick=time.monotonic()\\n    with ThreadPoolExecutor(max_workers=2) as pool:rows=list(pool.map(lambda i:read_one(root,train_ids[i]),chosen))\\n    elapsed=time.monotonic()-tick;diags=[r[1] for r in rows]\\n    thermal=np.array([r[0][\'thermal\'] for r in rows])\\n    varying=sum(len(np.unique(col[np.isfinite(col)]))>1 for col in thermal.T)\\n    supported=sum(d[\'observable_fraction\']>=.1 for d in diags)\\n    projection=elapsed*4439/len(rows)\\n    passed=supported>=8 and varying>=2 and projection<=2700\\n    receipt={\'schema\':1,\'experiment\':\'label_blind_sparse_profile\',\'chips\':len(rows),\'seconds\':elapsed,\\n        \'workers\':2,\'linear_full_pass_projection_seconds\':projection,\'supported_chips\':supported,\\n        \'varying_thermal_columns\':int(varying),\'summary\':summarize(diags),\'passed\':bool(passed),\\n        \'limits\':[\'Projection is not a runtime guarantee. No labels or accuracy inspected.\',\\n                  \'Reflectance-shaped anomalies are not calibrated heat/fire measurements.\']}\\n    (output/\'profile_receipt.json\').write_text(json.dumps(receipt,indent=2,allow_nan=False))\\n    # Only private relative-shape overlays, selected without targets; never publish them.\\n    import matplotlib.pyplot as plt\\n    for case,i in enumerate(chosen[:6]):\\n        import rasterio\\n        with warnings.catch_warnings():\\n            warnings.simplefilter(\'ignore\')\\n            with rasterio.open(Path(root)/\'images\'/f\'{train_ids[i]}.tif\') as src:chip=src.read()\\n        r,observed,clear,cloud=hotspots.layers(chip)\\n        _,mask=hotspots.thermal(r,observed,clear,cloud)\\n        rgb=np.moveaxis(chip[[3,2,1]].astype(float)*1e-4,0,-1)\\n        scale=max(float(np.quantile(rgb,.995)),.01)\\n        fig,axes=plt.subplots(1,2,figsize=(8,3.5))\\n        axes[0].imshow(np.clip(rgb/scale,0,1));axes[0].set_title(f\'Private label-blind case {case+1}\')\\n        axes[1].imshow(np.clip(rgb/scale,0,1));ys,xs=np.where(mask);axes[1].scatter(xs,ys,s=8,facecolors=\'none\',edgecolors=\'red\')\\n        axes[1].set_title(\'Sparse spectral candidates; not fire truth\')\\n        for ax in axes:ax.axis(\'off\')\\n        fig.tight_layout();fig.savefig(output/f\'private_overlay_{case+1}.png\',dpi=110);plt.close(fig)\\n    print(\'SPECTRAL_PROFILE_BEGIN\');print(json.dumps(receipt,sort_keys=True));print(\'SPECTRAL_PROFILE_END\',flush=True)\\n    if not passed:raise RuntimeError(\'Fixed sparse-profile feasibility gate failed; no automatic threshold adjustment\')\\n    return receipt\\n\\ndef matrices(root,ids,cache):\\n    cache=Path(cache);source=inspect.getsource(hotspots)+inspect.getsource(features)+inspect.getsource(read_one)\\n    source_sha=hashlib.sha256(source.encode()).hexdigest()\\n    fingerprint=hashlib.sha256((source_sha+\'\\\\n\'+str(Path(root).resolve())+\'\\\\n\'+\'\\\\n\'.join(ids)).encode()).hexdigest()\\n    widths={\'official\':54,\'thermal\':12,\'quality\':4,\'brightness\':6,\'scrambled\':12}\\n    if cache.exists():\\n        with np.load(cache,allow_pickle=False) as saved:\\n            if saved[\'fingerprint\'].item()==fingerprint:\\n                result={name:saved[name].copy() for name in widths}\\n                if any(a.shape!=(len(ids),widths[n]) or np.isinf(a).any() for n,a in result.items()):raise ValueError(\'Invalid private feature cache\')\\n                print(\'Verified sparse private cache loaded.\',flush=True)\\n                return result,json.loads(saved[\'receipt\'].item())\\n    tick=time.monotonic();rows={name:[] for name in widths};diags=[]\\n    with ThreadPoolExecutor(max_workers=2) as pool:\\n        for count,(values,d) in enumerate(pool.map(lambda sid:read_one(root,sid),ids),1):\\n            for name in widths:rows[name].append(values[name])\\n            diags.append(d)\\n            if time.monotonic()-tick>2700:raise RuntimeError(\'Extraction exceeded45minutes\')\\n            if count%250==0 or count==len(ids):print(\'SPARSE_CHIPS\',count,len(ids),round(time.monotonic()-tick,1),flush=True)\\n    result={name:np.asarray(value,dtype=float) for name,value in rows.items()}\\n    if any(a.shape!=(len(ids),widths[n]) or np.isinf(a).any() for n,a in result.items()):raise ValueError(\'Invalid feature matrix\')\\n    receipt={\'schema\':1,\'chips\':len(ids),\'widths\':widths,\'source_sha256\':source_sha,\'seconds\':time.monotonic()-tick,\\n        \'summary\':summarize(diags),\'names\':{\'thermal\':THERMAL_NAMES,\'quality\':QUALITY_NAMES,\'brightness\':BRIGHTNESS_NAMES,\\n        \'official\':sorted(features.chip_summary(np.ones((14,16,16),dtype=np.uint16),\'official\'))},\\n        \'limits\':[\'L2A content-only descriptors; unknown physical pixel area. No labels used in extraction.\']}\\n    cache.parent.mkdir(parents=True,exist_ok=True)\\n    np.savez_compressed(cache,**result,fingerprint=np.array(fingerprint),receipt=np.array(json.dumps(receipt,allow_nan=False)))\\n    return result,receipt\\n", "spectral_lab/features.py": "\\"\\"\\"Content-only chip summaries; explicit cloud ablations, no geospatial metadata.\\n\\nThe official feature recipe is adapted from Zelzy\'s Apache-2.0 starter.\\nOpenAI Codex assisted the implementation and tests.\\n\\"\\"\\"\\nimport numpy as np\\n\\nBAND_ORDER = [\'B1\',\'B2\',\'B3\',\'B4\',\'B5\',\'B6\',\'B7\',\'B8\',\'B8A\',\'B9\',\'B11\',\'B12\',\'SCL\',\'CLDPRB\']\\nSTAT_BANDS = [\'B2\',\'B3\',\'B4\',\'B8\',\'B11\',\'B12\']\\nSCL_CLASSES = [2,3,4,5,6,7,8,9,10,11]\\nMIN_CLEAR = 500\\n\\n\\ndef _statistics(values, prefix):\\n    if not values.size:\\n        return {f\'{prefix}_{name}\': float(\'nan\') for name in [\'mean\',\'std\',\'p50\',\'p95\',\'p99\',\'max\']}\\n    p50,p95,p99 = np.percentile(values,[50,95,99])\\n    return {f\'{prefix}_{name}\':float(value) for name,value in\\n            zip([\'mean\',\'std\',\'p50\',\'p95\',\'p99\',\'max\'],[values.mean(),values.std(),p50,p95,p99,values.max()])}\\n\\n\\ndef _spatial(bands, mask, sufficient):\\n    names = [f\'s_{band}_{stat}\' for band in [\'B11\',\'B12\'] for stat in [\'centre_mean\',\'outer_mean\',\'centre_minus_outer\']]\\n    names += [\'s_ndvi_centre_minus_outer\',\'s_bsi_centre_minus_outer\',\'s_B12_tail_contrast\',\'s_B11_tail_contrast\',\\n              \'s_B12_centre_std\',\'s_B12_gradient\',\'s_B11_gradient\',\'s_ndvi_gradient\',\'s_centre_clear_fraction\',\'s_outer_clear_fraction\']\\n    if not sufficient:\\n        return dict.fromkeys(names,float(\'nan\'))\\n    height,width = mask.shape\\n    rows,cols = np.indices(mask.shape)\\n    centre = (rows>=height/4)&(rows<3*height/4)&(cols>=width/4)&(cols<3*width/4)\\n    c,edge = mask&centre,mask&~centre\\n    output = {}\\n    for band in [\'B11\',\'B12\']:\\n        a = bands[band]\\n        cm = float(a[c].mean()) if c.sum()>=20 else float(\'nan\')\\n        em = float(a[edge].mean()) if edge.sum()>=20 else float(\'nan\')\\n        output.update({f\'s_{band}_centre_mean\':cm,f\'s_{band}_outer_mean\':em,f\'s_{band}_centre_minus_outer\':cm-em})\\n    ndvi = (bands[\'B8\']-bands[\'B4\'])/(bands[\'B8\']+bands[\'B4\']+1e-6)\\n    bsi = ((bands[\'B11\']+bands[\'B4\'])-(bands[\'B8\']+bands[\'B2\']))/(bands[\'B11\']+bands[\'B4\']+bands[\'B8\']+bands[\'B2\']+1e-6)\\n    for label,a in [(\'ndvi\',ndvi),(\'bsi\',bsi)]:\\n        output[f\'s_{label}_centre_minus_outer\'] = float(a[c].mean()-a[edge].mean()) if min(c.sum(),edge.sum())>=20 else float(\'nan\')\\n    for band in [\'B12\',\'B11\']:\\n        p50,p99 = np.percentile(bands[band][mask],[50,99])\\n        output[f\'s_{band}_tail_contrast\'] = float(p99-p50)\\n    output[\'s_B12_centre_std\'] = float(bands[\'B12\'][c].std()) if c.sum()>=20 else float(\'nan\')\\n    for label,a in [(\'B12\',bands[\'B12\']),(\'B11\',bands[\'B11\']),(\'ndvi\',ndvi)]:\\n        vertical = np.abs(np.diff(a,axis=0))[mask[1:]&mask[:-1]]\\n        horizontal = np.abs(np.diff(a,axis=1))[mask[:,1:]&mask[:,:-1]]\\n        values = np.concatenate([vertical,horizontal])\\n        output[f\'s_{label}_gradient\'] = float(values.mean()) if values.size>=20 else float(\'nan\')\\n    output[\'s_centre_clear_fraction\'] = float(mask[centre].mean())\\n    output[\'s_outer_clear_fraction\'] = float(mask[~centre].mean())\\n    return output\\n\\n\\ndef chip_summary(chip, method=\'official\', spatial=False):\\n    chip = np.asarray(chip)\\n    if chip.ndim!=3 or chip.shape[0]!=14 or min(chip.shape[1:])<8:\\n        raise ValueError(\'Expected fourteen chip layers and at least eight pixels per dimension\')\\n    if method not in {\'official\',\'no_fallback\',\'strict\'}:\\n        raise ValueError(\'Unknown cloud-mask method\')\\n    if not np.isfinite(chip).all():\\n        raise ValueError(\'Unexpected nonfinite source chip\')\\n    scl,cloud = chip[12],chip[13]\\n    has_data = chip[3]>0\\n    clear = has_data&~np.isin(scl,[3,8,9,10])\\n    if method==\'strict\': clear = has_data&np.isin(scl,[4,5,6])&(cloud<20)\\n    sufficient = clear.sum()>=MIN_CLEAR\\n    mask = has_data if method==\'official\' and not sufficient else clear\\n    bands = {name:chip[index].astype(np.float64)*1e-4 for index,name in enumerate(BAND_ORDER[:12])}\\n    available = bool(mask.any()) if method==\'official\' else bool(sufficient)\\n    output = {}\\n    for band in STAT_BANDS:\\n        values = bands[band][mask] if available else np.array([])\\n        if method==\'official\' and not values.size: values=np.zeros(1)\\n        output.update(_statistics(values,f\'c_{band}\'))\\n    if available:\\n        b4,b8,b11,b12 = (bands[name][mask] for name in [\'B4\',\'B8\',\'B11\',\'B12\'])\\n        ndvi = (b8-b4)/(b8+b4+1e-6)\\n        values = [ndvi.std(),np.percentile(ndvi,95),np.percentile(b12/(b11+1e-6),95),\\n                  (b12>.2).mean(),(b12>.4).mean(),(b11>.4).mean()]\\n    else:\\n        values = [0.]*6 if method==\'official\' else [float(\'nan\')]*6\\n    output.update(dict(zip([\'c_ndvi_std\',\'c_ndvi_p95\',\'c_b12_b11_p95\',\'c_hot12_gt_0.2\',\'c_hot12_gt_0.4\',\'c_hot11_gt_0.4\'],map(float,values))))\\n    for cls in SCL_CLASSES: output[f\'c_scl_{cls}\']=float((scl==cls).mean())\\n    output[\'c_cloudprob\']=float(cloud.mean())\\n    output[\'c_valid_frac\']=float(mask.mean())\\n    if method!=\'official\':\\n        output[\'q_insufficient_clear\']=float(not sufficient)\\n        output[\'q_clear_count\']=float(clear.sum())\\n        output[\'q_cloud_high_fraction\']=float((cloud>=40).mean())\\n    if spatial: output.update(_spatial(bands,mask,available))\\n    return output\\n", "spectral_lab/hotspots.py": "\\"\\"\\"Independent physics-inspired descriptors, not calibrated fire detections.\\n\\nMotivated by Massimetti2020, Fisher/Wooster2018 and the public methods reviewed\\nin literature-20261123. No third-party implementation is vendored.\\n\\"\\"\\"\\nimport numpy as np\\nfrom scipy.ndimage import uniform_filter,label\\n\\nEPS=1e-10\\nSTRUCTURE_NAMES=[\'peak\',\'top8_mean\',\'selected_sum\',\'selected_count\',\'components8\',\'largest_component_fraction\']\\nTHERMAL_NAMES=[f\'h_{n}\' for n in STRUCTURE_NAMES]+[\\n    \'h_mean_positive_e12\',\'h_max_positive_e11\',\'h_median_nir_excess_ratio\',\\n    \'h_low_nir_fraction\',\'h_candidate_nonclear_fraction\',\'h_background_support_fraction\']\\nQUALITY_NAMES=[\'q_observable_fraction\',\'q_clear_fraction\',\'q_valid_cloud_mean\',\'q_defect_flag_fraction\']\\nBRIGHTNESS_NAMES=[f\'b_{n}\' for n in STRUCTURE_NAMES]\\n\\ndef validate(chip):\\n    chip=np.asarray(chip)\\n    if chip.ndim!=3 or chip.shape[0]!=14 or min(chip.shape[1:])<15:raise ValueError(\'Expected14layers and >=15pixels\')\\n    if not np.isfinite(chip).all() or (chip<0).any():raise ValueError(\'Invalid nonfinite/negative source pixel\')\\n    if (chip[13]>100).any() or (chip[12]>11).any():raise ValueError(\'Quality layer scale mismatch\')\\n    if not np.equal(chip[12],np.floor(chip[12])).all():raise ValueError(\'Noninteger scene classes\')\\n    return chip\\n\\ndef layers(chip):\\n    # B8A is zero-based8; B11=10,B12=11. Float BEFORE subtraction.\\n    r=chip[[8,10,11]].astype(np.float64)*1e-4\\n    scl=chip[12];cloud=chip[13].astype(float)*.01\\n    observed=(r.sum(axis=0)>0)&~np.isin(scl,[0,1])\\n    clear=observed&~np.isin(scl,[3,8,9,10])&(cloud<.4)\\n    return r,observed,clear,cloud\\n\\ndef local_mean(values,valid):\\n    total=uniform_filter(np.where(valid,values,0.),size=15,mode=\'constant\',cval=0.)\\n    support=uniform_filter(valid.astype(float),size=15,mode=\'constant\',cval=0.)\\n    return np.divide(total,support,out=np.zeros_like(total),where=support>0)\\n\\ndef select(score,eligible):\\n    idx=np.flatnonzero(eligible&(score>EPS))\\n    if len(idx):idx=idx[np.lexsort((idx,-score.ravel()[idx]))[:32]]\\n    selected=np.zeros(score.shape,dtype=bool);selected.ravel()[idx]=True\\n    return idx,selected\\n\\ndef structure(score,idx,selected,has_observations):\\n    if not has_observations:return np.full(6,np.nan)\\n    if not len(idx):return np.array([np.nan,np.nan,0.,0.,0.,np.nan])\\n    values=score.ravel()[idx]\\n    components,n=label(selected,structure=np.ones((3,3),dtype=np.uint8))\\n    largest=np.bincount(components.ravel())[1:].max()\\n    return np.array([values.max(),values[:8].mean(),values.sum(),len(idx),n,largest/len(idx)],dtype=float)\\n\\ndef thermal(r,observed,clear,cloud):\\n    r8a,r11,r12=r\\n    positive=np.maximum((r12-r11)/np.maximum(r8a,.02),0.)\\n    score=positive-local_mean(positive,observed)\\n    curvature=r12-2*r11+r8a\\n    idx,selected=select(score,observed&(curvature>EPS))\\n    first=structure(score,idx,selected,bool(observed.any()))\\n    if not len(idx):return np.r_[first,np.full(6,np.nan)],selected\\n    height,width=observed.shape;excess=[]\\n    for flat in idx:\\n        row,col=divmod(int(flat),width)\\n        top,bottom=max(row-7,0),min(row+8,height);left,right=max(col-7,0),min(col+8,width)\\n        background=(clear&~selected)[top:bottom,left:right].copy()\\n        ys,xs=np.ogrid[top:bottom,left:right]\\n        background[(abs(ys-row)<=1)&(abs(xs-col)<=1)]=False\\n        if background.sum()>=16:\\n            local=r[:,top:bottom,left:right][:,background]\\n            excess.append(r[:,row,col]-np.median(local,axis=1))\\n    cloudy=float(np.mean(~clear.ravel()[idx]));support=len(excess)/len(idx)\\n    if excess:\\n        e=np.asarray(excess);positive12=e[:,2]>0\\n        ratio=float(np.median(e[positive12,0]/np.maximum(e[positive12,2],.02))) if positive12.any() else np.nan\\n        low=float(np.mean(e[positive12,0]<.1*e[positive12,2])) if positive12.any() else np.nan\\n        context=[np.maximum(e[:,2],0).mean(),np.maximum(e[:,1],0).max(),ratio,low,cloudy,support]\\n    else:context=[np.nan,np.nan,np.nan,np.nan,cloudy,support]\\n    return np.r_[first,context],selected\\n\\ndef extract(chip,include_scramble=True):\\n    chip=validate(chip);r,observed,clear,cloud=layers(chip)\\n    h,selected=thermal(r,observed,clear,cloud)\\n    bright=r[2]-local_mean(r[2],observed)\\n    bi,bmask=select(bright,observed)\\n    b=structure(bright,bi,bmask,bool(observed.any()))\\n    q=np.array([observed.mean(),clear.mean(),cloud[observed].mean() if observed.any() else np.nan,(chip[12]==1).mean()])\\n    result={\'thermal\':h,\'quality\':q,\'brightness\':b}\\n    if include_scramble:\\n        perm=np.random.default_rng(2026).permutation(observed.size)\\n        shape=observed.shape\\n        sr=r.reshape(3,-1)[:,perm].reshape(r.shape)\\n        so=observed.ravel()[perm].reshape(shape);sc=clear.ravel()[perm].reshape(shape);sp=cloud.ravel()[perm].reshape(shape)\\n        result[\'scrambled\']=thermal(sr,so,sc,sp)[0]\\n    if any(np.isinf(value).any() for value in result.values()):raise ValueError(\'Infinite extracted feature\')\\n    diagnostic={\'observable_fraction\':float(observed.mean()),\'clear_fraction\':float(clear.mean()),\\n        \'candidate_count\':int(selected.sum()),\'strict_clear_candidate_count\':int((selected&clear).sum()),\\n        \'band_over_one_fraction\':float((r>1).mean()),\'band_max\':float(r.max()),\\n        \'quality_flag1_fraction\':float((chip[12]==1).mean())}\\n    return result,diagnostic\\n", "spectral_lab/independent_audit.py": "\\"\\"\\"Recompute private OOF evidence; output aggregates only, never rows or site keys.\\"\\"\\"\\nfrom pathlib import Path\\nimport hashlib\\nimport json\\nimport numpy as np\\nfrom sklearn.model_selection import GroupShuffleSplit\\nfrom .core import TARGETS,load_tables,site_folds\\n\\ndef independent_score(y,p,weights=None):\\n    y=np.asarray(y,dtype=float);p=np.asarray(p,dtype=float)\\n    if y.shape!=p.shape or y.ndim!=2 or y.shape[1]!=5 or not np.isfinite(p).all() or np.isinf(y).any():\\n        raise ValueError(\'Invalid audit prediction/label contract\')\\n    weights=np.ones(len(y)) if weights is None else np.asarray(weights,dtype=float)\\n    if weights.shape!=(len(y),) or not np.isfinite(weights).all() or (weights<0).any():raise ValueError(\'Invalid audit weights\')\\n    errors={};counts={}\\n    for j,name in enumerate(TARGETS):\\n        keep=np.isfinite(y[:,j]);den=float(weights[keep].sum())\\n        if den<=0:raise ValueError(\'Missing observed labels in audit target\')\\n        errors[name]=float(np.sqrt(np.sum(weights[keep]*(y[keep,j]-p[keep,j])**2)/den));counts[name]=den\\n    return {\'mcrmse\':float(np.mean(list(errors.values()))),\'rmse\':errors,\'observed\':counts}\\n\\ndef paired_site_summary(y,control,candidate,groups,draws=1000):\\n    groups=np.asarray(groups);sites,inverse=np.unique(groups,return_inverse=True)\\n    if len(groups)!=len(y):raise ValueError(\'Mismatched audit groups\')\\n    point=independent_score(y,control)[\'mcrmse\']-independent_score(y,candidate)[\'mcrmse\']\\n    rng=np.random.default_rng(2026);deltas=[]\\n    for _ in range(draws):\\n        weights=np.bincount(rng.integers(0,len(sites),len(sites)),minlength=len(sites))[inverse]\\n        try:deltas.append(independent_score(y,control,weights)[\'mcrmse\']-independent_score(y,candidate,weights)[\'mcrmse\'])\\n        except ValueError:continue\\n    if len(deltas)<.99*draws:raise ValueError(\'Insufficient valid site bootstrap draws\')\\n    macro={};support={}\\n    for j,t in enumerate(TARGETS):\\n        scores={name:[] for name in [\'control\',\'candidate\']}\\n        for site in sites:\\n            keep=(groups==site)&np.isfinite(y[:,j])\\n            if keep.any():\\n                for name,p in [(\'control\',control),(\'candidate\',candidate)]:scores[name].append(float(np.sqrt(np.mean((y[keep,j]-p[keep,j])**2))))\\n        macro[t]={name:float(np.mean(v)) for name,v in scores.items()};support[t]=len(scores[\'control\'])\\n    return {\'sites\':len(sites),\'point_gain\':point,\'conditional_site_bootstrap_95_percentile\':np.quantile(deltas,[.025,.975]).tolist(),\\n        \'valid_draws\':len(deltas),\'site_macro_rmse\':macro,\'label_bearing_sites\':support,\\n        \'limitation\':\'Fixed OOF predictions only; no retraining/selection uncertainty, independent confirmation or temporal claim.\'}\\n\\ndef audit(root,output):\\n    output=Path(output);receipt=json.loads((output/\'foundation_receipt.json\').read_text())\\n    train,test,sample,names=load_tables(root);groups=train.site_key.to_numpy();y=train[TARGETS].to_numpy(dtype=float)\\n    dev,_=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=2026).split(train[names],groups=groups))\\n    saved=np.load(output/\'private_development_predictions.npz\',allow_pickle=False)\\n    if not np.array_equal(saved[\'truth\'],y[dev],equal_nan=True):raise ValueError(\'Saved OOF truth does not match exact development rows\')\\n    scores={name:independent_score(y[dev],saved[name]) for name in [\'control\',\'candidate\',\'raw_foundation\']}\\n    for name,value in scores.items():\\n        stated=receipt[\'comparison\'][\'scores\'][name]\\n        if not np.isclose(value[\'mcrmse\'],stated[\'mcrmse\'],atol=1e-12,rtol=0):raise ValueError(\'Receipt MCRMSE disagrees with independent calculation\')\\n        for t in TARGETS:\\n            if not np.isclose(value[\'rmse\'][t],stated[\'rmse\'][t],atol=1e-12,rtol=0) or value[\'observed\'][t]!=stated[\'observed\'][t]:\\n                raise ValueError(\'Receipt target score/denominator disagrees with independent calculation\')\\n    mean=np.zeros_like(y[dev])\\n    for a,b in site_folds(groups[dev],3,seed=2026):mean[b]=np.nanmean(y[dev][a],axis=0)\\n    result={\'schema\':1,\'experiment\':receipt[\'experiment\'],\'independent_scores\':scores,\\n        \'zero\':independent_score(y[dev],np.zeros_like(mean)),\'outer_fold_mean\':independent_score(y[dev],mean),\\n        \'paired_sites\':paired_site_summary(y[dev],saved[\'control\'],saved[\'candidate\'],groups[dev]),\\n        \'exact_development_truth_match\':True,\'no_test_label_access\':True,\'rows_exported_in_audit\':False}\\n    submission=receipt.get(\'submission\')\\n    if submission:\\n        import pandas as pd\\n        path=output/submission[\'file\'];table=pd.read_csv(path,dtype={\'sample_id\':str})\\n        if list(table.columns)!=[\'sample_id\',*TARGETS] or not table.sample_id.equals(sample.sample_id):raise ValueError(\'CSV schema/order mismatch\')\\n        if len(table)!=len(test) or table.sample_id.duplicated().any() or not np.isfinite(table[TARGETS].to_numpy()).all():raise ValueError(\'CSV coverage/finiteness mismatch\')\\n        digest=hashlib.sha256(path.read_bytes()).hexdigest()\\n        if digest!=submission[\'sha256\']:raise ValueError(\'CSV hash mismatch\')\\n        result[\'verified_submission\']={\'sha256\':digest,\'rows\':len(table),\'columns\':list(table.columns)}\\n    (output/\'independent_audit.json\').write_text(json.dumps(result,indent=2,allow_nan=False))\\n    print(\'INDEPENDENT_AUDIT_BEGIN\');print(json.dumps(result,sort_keys=True));print(\'INDEPENDENT_AUDIT_END\')\\n    return result\\n\\n", "spectral_lab/selection.py": "\\"\\"\\"Target-wise conservative mixtures with selection confined to inner sites.\\"\\"\\"\\nimport time\\nimport numpy as np\\nfrom .core import TARGETS, metric, site_folds\\nfrom .baseline import fit_predict\\n\\nAMPLITUDES = [0.,.25,.5,.75,1.]\\nIMAGE_WEIGHTS = [0.,.5,1.]\\n\\n\\ndef mix_predictions(base, enhanced, mean, amplitude, image_weight):\\n    if not (0<=amplitude<=1 and 0<=image_weight<=1):\\n        raise ValueError(\'Only convex prediction mixtures are permitted\')\\n    base,enhanced,mean = map(np.asarray,[base,enhanced,mean])\\n    return mean+amplitude*((1-image_weight)*(base-mean)+image_weight*(enhanced-mean))\\n\\n\\ndef choose_mix(truth, base, enhanced, means, image_weights=None):\\n    y,b,e,m = map(lambda a:np.asarray(a,dtype=float),[truth,base,enhanced,means])\\n    if any(a.shape!=y.shape for a in [b,e,m]) or not all(np.isfinite(a).all() for a in [b,e,m]):\\n        raise ValueError(\'Mixture inputs must have matching shapes and finite predictions\')\\n    keep = np.isfinite(y)\\n    if np.isinf(y).any() or keep.sum()<2:\\n        raise ValueError(\'Need at least two measured, finite tuning labels\')\\n    best=None\\n    for w in (IMAGE_WEIGHTS if image_weights is None else image_weights):\\n        for a in AMPLITUDES:\\n            p=mix_predictions(b,e,m,a,w)\\n            error=float(np.sqrt(np.mean((y[keep]-p[keep])**2)))\\n            key=(error,w,a)\\n            if best is None or error<best[0][0]-1e-12 or (abs(error-best[0][0])<=1e-12 and (w,a)<best[0][1:]):\\n                best=(key,{\'amplitude\':a,\'image_weight\':w,\'inner_rmse\':error})\\n    return best[1]\\n\\n\\ndef raw_oof(x,y,groups,n_splits=3,params=None):\\n    y=np.asarray(y,dtype=float)\\n    predictions=np.full(y.shape,np.nan)\\n    means=np.full(y.shape,np.nan)\\n    for fit,check in site_folds(groups,n_splits):\\n        for j in range(5):\\n            _,predictions[check,j]=fit_predict(x[fit],y[fit,j],x[check],params)\\n            means[check,j]=np.nanmean(y[fit,j])\\n    if not np.isfinite(predictions).all() or not np.isfinite(means).all():\\n        raise AssertionError(\'Incomplete inner cross-validation\')\\n    return predictions,means\\n\\n\\ndef nested_compare(features,truth,groups,folds=None,inner_splits=3,params=None):\\n    y=np.asarray(truth,dtype=float)\\n    groups=np.asarray(groups)\\n    features={key:np.asarray(value,dtype=float) for key,value in features.items()}\\n    if \'table\' not in features or any(len(value)!=len(y) for value in features.values()):\\n        raise ValueError(\'Features require the shared table reference and matching rows\')\\n    folds=site_folds(groups,3) if folds is None else folds\\n    predictions={key:np.full(y.shape,np.nan) for key in features}\\n    raw={key:np.full(y.shape,np.nan) for key in features}\\n    means=np.full(y.shape,np.nan)\\n    reports=[]\\n    seen=np.zeros(len(y),dtype=int)\\n    for number,(fit,check) in enumerate(folds,1):\\n        tick=time.monotonic()\\n        if set(groups[fit])&set(groups[check]) or np.intersect1d(fit,check).size:\\n            raise ValueError(\'Outer facility leakage\')\\n        seen[check]+=1\\n        inner={}\\n        for name,x in features.items():\\n            inner[name],inner_means=raw_oof(x[fit],y[fit],groups[fit],inner_splits,params)\\n        for name,x in features.items():\\n            for j in range(5):\\n                _,raw[name][check,j]=fit_predict(x[fit],y[fit,j],x[check],params)\\n        means[check]=np.nanmean(y[fit],axis=0)\\n        choices={}\\n        for name in features:\\n            choices[name]=[]\\n            for j in range(5):\\n                choice=choose_mix(y[fit,j],inner[\'table\'][:,j],inner[name][:,j],inner_means[:,j],\\n                                  image_weights=[0.] if name==\'table\' else None)\\n                choices[name].append(choice)\\n                predictions[name][check,j]=mix_predictions(raw[\'table\'][check,j],raw[name][check,j],\\n                    means[check,j],choice[\'amplitude\'],choice[\'image_weight\'])\\n        reports.append({\'fold\':number,\'fit_sites\':len(set(groups[fit])),\'check_sites\':len(set(groups[check])),\\n                        \'choices\':choices,\'scores\':{name:metric(y[check],p[check]) for name,p in predictions.items()},\\n                        \'seconds\':round(time.monotonic()-tick,3)})\\n        print(\'Outer fold\',number,\'scores:\',{name:round(value[\'mcrmse\'],6) for name,value in reports[-1][\'scores\'].items()},flush=True)\\n    if not np.all(seen==1):\\n        raise ValueError(\'Outer folds must hold every row out exactly once\')\\n    scores={name:metric(y,p) for name,p in predictions.items()}\\n    return {\'scores\':scores,\'folds\':reports,\'predictions\':predictions,\'raw_predictions\':raw,\\n            \'mean_predictions\':means,\'fold_mean_score\':metric(y,means),\\n            \'raw_scores\':{name:metric(y,p) for name,p in raw.items()}}\\n\\n\\ndef final_predict(train_features,test_features,truth,groups,variant,n_splits=5,params=None):\\n    y=np.asarray(truth,dtype=float)\\n    names=[\'table\'] if variant==\'table\' else [\'table\',variant]\\n    oof={}\\n    test_predictions={}\\n    for name in names:\\n        oof[name],means=raw_oof(np.asarray(train_features[name]),y,groups,n_splits,params)\\n        test_predictions[name]=np.zeros((len(test_features[name]),5))\\n        for j in range(5):\\n            _,test_predictions[name][:,j]=fit_predict(train_features[name],y[:,j],test_features[name],params)\\n    mu=np.nanmean(y,axis=0)\\n    result=np.zeros_like(test_predictions[\'table\'])\\n    choices=[]\\n    for j in range(5):\\n        choice=choose_mix(y[:,j],oof[\'table\'][:,j],oof[variant][:,j],means[:,j],\\n                          image_weights=[0.] if variant==\'table\' else None)\\n        result[:,j]=mix_predictions(test_predictions[\'table\'][:,j],test_predictions[variant][:,j],\\n                                    mu[j],choice[\'amplitude\'],choice[\'image_weight\'])\\n        choices.append(choice)\\n    if not np.isfinite(result).all(): raise ValueError(\'Nonfinite final prediction\')\\n    return result,choices\\n", "test_comparison.py": "import numpy as np\\nimport pandas as pd\\nimport pytest\\nfrom spectral_lab import comparison as c\\nfrom spectral_lab.selection import choose_mix,mix_predictions\\n\\ndef test_zero_branch_matches_incumbent_selection():\\n    rng=np.random.default_rng(10);y=rng.normal(size=90);b=rng.normal(size=90);e=rng.normal(size=90);m=np.zeros(90)\\n    # A duplicate official branch cannot improve the incumbent\'s grid.\\n    choice=c.choose_branch(y,b,e,e,m);old=choose_mix(y,b,e,m)\\n    assert np.allclose(c.branch_predict(choice,b,e,e,m),mix_predictions(b,e,m,old[\'amplitude\'],old[\'image_weight\']))\\n    assert c.choose_branch(np.zeros(90),b,e,e,m)[\'branch_weight\']==0\\n\\ndef fake_fit(x,y,z):\\n    return np.full(len(z),np.nanmean(y))+.01*np.nan_to_num(z[:,0])\\n\\ndef test_outer_labels_cannot_change_predictions_and_four_targets_protected():\\n    rng=np.random.default_rng(22);g=np.repeat(np.arange(12),4);y=rng.normal(size=(48,5));x=rng.normal(size=(48,3))\\n    f={name:x.copy() for name in [\'table\',\'official\',*c.VARIANTS]};fit=np.arange(36);check=np.arange(36,48)\\n    pred,choices,_=c.outer(f,y,g,fit,check,fake_fit)\\n    other=y.copy();other[check]=1e6\\n    altered,changed,_=c.outer(f,other,g,fit,check,fake_fit)\\n    assert choices==changed\\n    for name in pred:\\n        assert np.array_equal(pred[name],altered[name])\\n        assert np.array_equal(pred[name][:,:4],pred[\'C\'][:,:4])\\n    assert abs(c.metric(y[check],pred[\'C\'])[\'mcrmse\']-c.metric(y[check],pred[\'F\'])[\'mcrmse\']-(c.metric(y[check],pred[\'C\'])[\'rmse\'][\'t_firms\']-c.metric(y[check],pred[\'F\'])[\'rmse\'][\'t_firms\'])/5)<1e-12\\n\\ndef test_environment_partition_is_label_free_and_site_disjoint():\\n    rng=np.random.default_rng(17);groups=np.repeat(np.arange(16),3)\\n    columns=[\'temperature\',\'humidity\',\'wind_speed\',\'precipitation\',\'solar_rad\',\'elevation\',\'slope\']\\n    x=rng.normal(size=(48,7));x[24:]+=10;table=pd.DataFrame(x,columns=columns)\\n    folds,receipt=c.environment_folds(table,groups)\\n    table[\'t_firms\']=rng.normal(size=48);again,_=c.environment_folds(table,groups)\\n    assert not receipt[\'uses_targets\']\\n    assert min(receipt[\'site_counts\'])>=5\\n    for (a,b),(aa,bb) in zip(folds,again):\\n        assert np.array_equal(a,aa) and np.array_equal(b,bb)\\n        assert not set(groups[a])&set(groups[b])\\n\\ndef test_fit_only_missing_feature_guard():\\n    x=np.column_stack([np.arange(50),np.full(50,np.nan)]);y=np.sin(np.arange(50));z=np.array([[2.,1e5],[3.,-1e5]])\\n    first=c.safe_fit(x,y,z);z[:,1]*=1e4\\n    assert np.array_equal(first,c.safe_fit(x,y,z))\\n", "test_extractor.py": "import json\\nimport numpy as np\\nimport pytest\\nfrom spectral_lab import extractor as e\\n\\ndef test_cache_reuse_identity_and_corruption(tmp_path,monkeypatch):\\n    calls=[]\\n    def fake_read(root,sid):\\n        calls.append(sid)\\n        return {n:np.full(w,len(calls),float) for n,w in dict(official=54,thermal=12,quality=4,brightness=6,scrambled=12).items()},{\'support\':1.}\\n    monkeypatch.setattr(e,\'read_one\',fake_read)\\n    # Local function source is available, so source-aware cache can be tested.\\n    cache=tmp_path/\'cache.npz\';a,_=e.matrices(tmp_path,[\'a\',\'b\'],cache)\\n    again,_=e.matrices(tmp_path,[\'a\',\'b\'],cache)\\n    assert len(calls)==2 and np.array_equal(a[\'thermal\'],again[\'thermal\'])\\n    e.matrices(tmp_path,[\'b\',\'a\'],cache)\\n    assert len(calls)==4\\n    with np.load(cache,allow_pickle=False) as src:saved={n:src[n].copy() for n in src.files}\\n    saved[\'thermal\'][0,0]=np.inf;np.savez_compressed(cache,**saved)\\n    with pytest.raises(ValueError,match=\'Invalid private feature cache\'):e.matrices(tmp_path,[\'b\',\'a\'],cache)\\n", "test_hotspots.py": "import numpy as np\\nimport pytest\\nfrom spectral_lab.hotspots import extract\\n\\ndef scene(position=None,kind=\'hot\'):\\n    x=np.zeros((14,64,64),dtype=np.uint16);x[:12]=1000;x[8]=2000;x[10]=1500;x[11]=1000;x[12]=5\\n    if position:\\n        row,col=position;sl=(slice(row,row+7),slice(col,col+7))\\n        spectrum=(2100,4000,8000) if kind==\'hot\' else (6000,7000,8000)\\n        for band,v in zip([8,10,11],spectrum):x[band][sl]=v\\n    return x\\n\\ndef test_equal_b12_brightness_does_not_imply_equal_thermal_features():\\n    hot,reflect=scene((25,25)),scene((25,25),\'reflective\')\\n    assert np.array_equal(hot[11],reflect[11])\\n    h,_=extract(hot);r,_=extract(reflect)\\n    assert h[\'thermal\'][3]==32 and r[\'thermal\'][3]==0\\n    assert h[\'thermal\'][9]==1 and h[\'thermal\'][6]>0\\n    np.testing.assert_allclose(h[\'brightness\'],r[\'brightness\'],equal_nan=True)\\n\\ndef test_anomaly_features_translate_without_center_assumption_and_ignore_b8():\\n    values=[extract(scene(pos))[0][\'thermal\'] for pos in [(25,25),(10,39),(39,10)]]\\n    for v in values[1:]:np.testing.assert_allclose(v,values[0],atol=1e-10,equal_nan=True)\\n    x=scene((25,25));x[7]=65535\\n    np.testing.assert_allclose(extract(x)[0][\'thermal\'],values[0],equal_nan=True)\\n\\ndef test_missing_support_distinct_from_observed_no_candidate_and_no_uint_underflow():\\n    flat=scene();obs,_=extract(flat);empty,_=extract(np.zeros_like(flat))\\n    assert obs[\'thermal\'][3]==0 and np.isnan(empty[\'thermal\']).all()\\n    assert obs[\'quality\'][0]==1 and empty[\'quality\'][0]==0\\n    assert not any(np.isinf(v).any() for v in obs.values())\\n    assert np.isnan(obs[\'thermal\'][0])  # B12<B11 cannot wrap into a huge positive anomaly.\\n\\ndef test_quality_flags_scramble_and_source_contract():\\n    x=scene((25,25));x[13,25:32,25:32]=100\\n    a,_=extract(x);b,_=extract(x)\\n    assert a[\'thermal\'][10]==1\\n    for name in a:np.testing.assert_array_equal(a[name],b[name])\\n    assert not np.allclose(a[\'thermal\'],a[\'scrambled\'],equal_nan=True)\\n    for bad in [x[:13],x.astype(float)*np.nan]:\\n        with pytest.raises(ValueError):extract(bad)\\n    x[13,0,0]=101\\n    with pytest.raises(ValueError):extract(x)\\n"}')
    hashes=json.loads('{"spectral_lab/__init__.py": "45a906a5fe23371ff72484a2beca952479553b189d6a1753838dd507014cc8e6", "spectral_lab/audit_saved.py": "d8f1aba1fd5f59ab62a79ee57b62cc5b14c47ec6482142216283c6d25331deb2", "spectral_lab/baseline.py": "29cf9fc30386c6ffc104edaf196c7c396e6c1742a998521c9e9c1ea78aa80b75", "spectral_lab/comparison.py": "1b7935bb2cd098755ace829258675336b1b0d6356ed7c01632c396019bdd28fa", "spectral_lab/core.py": "bfdb1e7400bc3c31ae6621c9ef34e9b1420c97f1859a5b72f022897ac9a3ed6a", "spectral_lab/extractor.py": "f158de5690301aaacbee1f2fe4a0ed4e4a7ab7474399b9aaa7aea0c3edff65c8", "spectral_lab/features.py": "936ca060ea7f6c0ab1802d36e0a01495d598f1b88a89f6148ea0bd3d2c940aee", "spectral_lab/hotspots.py": "6936e1a133fcf524e3f8bc8a440eede563e55d2554f69385d6d32e207af1d51d", "spectral_lab/independent_audit.py": "fb6c344ab4e856d4551bc3c0131f1384cbe8ad2ac930de04f4f3ab9793589fe0", "spectral_lab/selection.py": "f8f771be572ccc3d73c6af53e44fb35ab00e561304f21b1501b1556f6c0ec4c9", "test_comparison.py": "725e1aaabf6ffd5ed7c0748171ff6d4a60a076b8fc86265ffe66e1ba840fe922", "test_extractor.py": "811e601bc90ad5d34d0fdc3b393680052a5790f73bbffd07c5787b97c715013f", "test_hotspots.py": "c67427a0c4807174a8cf9b637f01f090f6d615bca94f7b8c55d3f1bfb21c71eb"}')
    workspace=Path('/kaggle/working/lavalabs_spectral_source');workspace.mkdir(exist_ok=True)
    for name,code in sources.items():
        if hashlib.sha256(code.encode()).hexdigest()!=hashes[name]:raise ValueError('Source freeze mismatch')
        path=workspace/name;path.parent.mkdir(parents=True,exist_ok=True);path.write_text(code)
    sys.path.insert(0,str(workspace))
    check=subprocess.run([sys.executable,'-m','pytest',str(workspace),'-q'],capture_output=True,text=True)
    print(check.stdout[-5000:]);print(check.stderr[-1000:])
    if check.returncode:raise RuntimeError('Spectral regression checks failed')
    paths=[p.parent for p in Path('/kaggle/input').rglob('train.csv') if (p.parent/'images').is_dir() and (p.parent/'test.csv').exists()]
    if len(paths)!=1:raise RuntimeError('Expected exactly one official dataset directory')
    root=paths[0]
    output=Path('/kaggle/working/lavalabs_private_spectral');output.mkdir(exist_ok=True)
    (output/'source_freeze.json').write_text(json.dumps(hashes,sort_keys=True,indent=2))
    from spectral_lab.extractor import profile
    # Only identifiers are read for the label-blind feasibility pass. No labels.
    ids=pd.read_csv(root/'train.csv',usecols=['sample_id'],dtype={'sample_id':str}).sample_id.tolist()
    profile_receipt=profile(root,ids,output)
    print('PROFILE_READY_FOR_PRIVATE_LABEL_BLIND_INSPECTION',flush=True)


    # Execute after the fixed label-blind profile and overlays are reviewed.
    from spectral_lab.comparison import run
    spectral_receipt=run(root,output)
    from spectral_lab.audit_saved import audit
    spectral_audit=audit(root,output)


    """Post-evaluation aggregate diagnostics only; never changes fitting/selection."""
    from pathlib import Path
    import json,hashlib
    import numpy as np
    from sklearn.model_selection import GroupShuffleSplit
    from spectral_lab.core import TARGETS,load_tables,write_json

    output=Path('/kaggle/working/lavalabs_private_spectral')
    train,_,_,names=load_tables(root);groups=train.site_key.to_numpy()
    dev,_=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=2026).split(train[names],groups=groups))
    with np.load(output/'private_development_predictions.npz',allow_pickle=False) as saved:
        y=saved['truth'].copy();control=saved['C'].copy();candidate=saved['F'].copy();site=saved['groups'].copy()
    with np.load(output/'private_features.npz',allow_pickle=False) as saved:
        quality=saved['quality'][:len(train)][dev].copy()
    if not np.array_equal(y,train[TARGETS].to_numpy(dtype=float)[dev],equal_nan=True):raise ValueError('Diagnostic row identity mismatch')

    def segment(mask):
        answer={'rows':int(mask.sum()),'targets':{}}
        for j,t in enumerate(TARGETS):
            keep=mask&np.isfinite(y[:,j]);n=int(keep.sum())
            answer['targets'][t]={'observed':n}
            if n:
                answer['targets'][t].update({name:float(np.sqrt(np.mean((p[keep,j]-y[keep,j])**2))) for name,p in [('C',control),('F',candidate)]})
        return answer

    firm=y[:,4];threshold=float(np.nanquantile(firm,.9));valid=np.isfinite(firm)
    floor=float(np.nanmin(firm));site_errors=[]
    for s in np.unique(site):
        keep=(site==s)&valid;site_errors.append(float(np.sum((firm[keep]-control[keep,4])**2)))
    total=sum(site_errors)
    report={'schema':1,'purpose':'Error accounting only; no model or threshold selection.',
            'fixed_clear_fraction_strata':{'below_10_percent':segment(quality[:,1]<.1),
                 '10_to_50_percent':segment((quality[:,1]>=.1)&(quality[:,1]<.5)),'at_least_50_percent':segment(quality[:,1]>=.5)},
            'firms_label_diagnostics':{'development_floor':floor,'floor_rows':int(np.sum(valid&np.isclose(firm,floor,rtol=0,atol=1e-10))),
                 'development_90_percentile':threshold,'strictly_above_90_percentile':segment(valid&(firm>threshold)),
                 'at_or_below_90_percentile':segment(valid&(firm<=threshold)),
                 'top_five_sites_share_of_control_firms_squared_error':sum(sorted(site_errors,reverse=True)[:5])/total if total else 0},
            'no_site_identifiers_or_rows_exported':True,
            'limits':['Strata have different denominators and cannot be ranked as comparable global scores.',
                      'Tail boundary uses development labels for descriptive accounting only.',
                      'Error concentration cannot establish the leader method or future-time accuracy.']}
    write_json(output/'error_accounting.json',report)
    print('SPECTRAL_ERROR_ACCOUNTING_BEGIN');print(json.dumps(report,sort_keys=True,allow_nan=False));print('SPECTRAL_ERROR_ACCOUNTING_END',flush=True)


    # Accepted full refit; model/feature comparison source remains unchanged.
    import json,hashlib,subprocess,sys
    sources=json.loads('{"spectral_lab/refit.py": "\\"\\"\\"Accepted FIRMS-only refit. Export remains private until fresh replay passes.\\"\\"\\"\\nfrom pathlib import Path\\nimport time,json,hashlib\\nimport numpy as np\\nfrom .core import TARGETS,load_tables,site_folds,export_submission,write_json\\nfrom .selection import choose_mix,mix_predictions\\nfrom .comparison import make_features,safe_fit,choose_branch,branch_predict\\nfrom .extractor import matrices\\n\\nEXPECTED_CONTROL=\'007abb84e2f53b01d7ca71929647d83ea6803e2cb186871fd61f3cda38892c20\'\\n\\ndef run(root,output):\\n    tick=time.monotonic();output=Path(output)\\n    decision=json.loads((output/\'spectral_receipt.json\').read_text())\\n    audit=json.loads((output/\'independent_spectral_audit.json\').read_text())\\n    if not decision[\'gate\'][\'passed\'] or not audit[\'passed\']:raise ValueError(\'Accepted audited comparison required\')\\n    train,test,sample,names=load_tables(root);y=train[TARGETS].to_numpy(dtype=float);groups=train.site_key.to_numpy()\\n    values,_=matrices(root,[*train.sample_id,*test.sample_id],output/\'private_features.npz\')\\n    x=np.vstack([train[names].to_numpy(dtype=float),test[names].to_numpy(dtype=float)])\\n    all_features=make_features(x,values);n=len(train);oof={k:np.full(y.shape,np.nan) for k in [\'table\',\'official\',\'mean\']};branch=np.full(n,np.nan)\\n    for number,(a,b) in enumerate(site_folds(groups,5),1):\\n        if time.monotonic()-tick>2700:raise RuntimeError(\'Accepted refit exceeded45minutes\')\\n        for name in [\'table\',\'official\']:\\n            for j in range(5):oof[name][b,j]=safe_fit(all_features[name][a],y[a,j],all_features[name][b])\\n        oof[\'mean\'][b]=np.nanmean(y[a],axis=0)\\n        branch[b]=safe_fit(all_features[\'F\'][a],y[a,4],all_features[\'F\'][b])\\n        print(\'SPECTRAL_REFIT_FOLD\',number,round(time.monotonic()-tick,2),flush=True)\\n    if not all(np.isfinite(a).all() for a in [*oof.values(),branch]):raise ValueError(\'Incomplete refit OOF\')\\n    choices=[choose_mix(y[:,j],oof[\'table\'][:,j],oof[\'official\'][:,j],oof[\'mean\'][:,j]) for j in range(5)]\\n    new_choice=choose_branch(y[:,4],oof[\'table\'][:,4],oof[\'official\'][:,4],branch,oof[\'mean\'][:,4])\\n    raw={name:np.column_stack([safe_fit(all_features[name][:n],y[:,j],all_features[name][n:]) for j in range(5)]) for name in [\'table\',\'official\']}\\n    new=safe_fit(all_features[\'F\'][:n],y[:,4],all_features[\'F\'][n:]);mu=np.nanmean(y,axis=0)\\n    control=np.column_stack([mix_predictions(raw[\'table\'][:,j],raw[\'official\'][:,j],mu[j],c[\'amplitude\'],c[\'image_weight\']) for j,c in enumerate(choices)])\\n    control_hash=export_submission(sample,control,output/\'private_control_replay.csv\')\\n    # The previous CSV is compared as a complete independent frozen identity.\\n    if control_hash!=EXPECTED_CONTROL:raise ValueError(\'Frozen final control CSV identity mismatch; do not submit\')\\n    candidate=control.copy();candidate[:,4]=branch_predict(new_choice,raw[\'table\'][:,4],raw[\'official\'][:,4],new,mu[4])\\n    if not np.array_equal(candidate[:,:4],control[:,:4]):raise AssertionError(\'Protected test targets changed\')\\n    candidate_hash=export_submission(sample,candidate,output/\'submission_spectral.csv\')\\n    np.savez_compressed(output/\'private_refit_predictions.npz\',truth=y,table=oof[\'table\'],official=oof[\'official\'],mean=oof[\'mean\'],branch=branch,control=control,candidate=candidate)\\n    result={\'schema\':1,\'experiment\':\'accepted_firms_only_full_refit\',\'rows\':len(test),\'train_rows\':n,\'sites\':len(set(groups)),\\n            \'control_sha256\':control_hash,\'candidate_sha256\':candidate_hash,\'protected_targets_byte_equal\':True,\\n            \'control_choices\':choices,\'firms_choice\':new_choice,\'seconds\':time.monotonic()-tick,\\n            \'scored_submission_made\':False,\'fresh_session_replay_verified\':False,\\n            \'limits\':[\'Full-training OOF selects final weights; it is not an untouched accuracy estimate.\',\\n                      \'Private final CSV; no scored export before fresh-session reproduction.\',\\n                      \'No physical fire, emission or spatial-mechanism claim follows from this score.\']}\\n    write_json(output/\'refit_receipt.json\',result)\\n    print(\'SPECTRAL_REFIT_BEGIN\');print(json.dumps(result,sort_keys=True));print(\'SPECTRAL_REFIT_END\',flush=True)\\n    return result\\n", "test_refit.py": "import json\\nimport numpy as np\\nimport pandas as pd\\nimport pytest\\nfrom spectral_lab import refit as r\\nfrom spectral_lab.core import TARGETS\\n\\ndef test_refit_refuses_failed_gate_before_reading_observations(tmp_path,monkeypatch):\\n    (tmp_path/\'spectral_receipt.json\').write_text(json.dumps({\'gate\':{\'passed\':False}}))\\n    (tmp_path/\'independent_spectral_audit.json\').write_text(json.dumps({\'passed\':True}))\\n    def forbidden(*args):raise AssertionError(\'Read observations despite failed gate\')\\n    monkeypatch.setattr(r,\'load_tables\',forbidden)\\n    with pytest.raises(ValueError,match=\'Accepted audited\'):r.run(tmp_path,tmp_path)\\n\\ndef test_refit_identity_mismatch_blocks_candidate_export(tmp_path,monkeypatch):\\n    (tmp_path/\'spectral_receipt.json\').write_text(json.dumps({\'gate\':{\'passed\':True}}))\\n    (tmp_path/\'independent_spectral_audit.json\').write_text(json.dumps({\'passed\':True}))\\n    rng=np.random.default_rng(10);n=60;names=[f\'x{i}\' for i in range(29)]\\n    train=pd.DataFrame(rng.normal(size=(n,29)),columns=names);test=pd.DataFrame(rng.normal(size=(4,29)),columns=names)\\n    train[\'sample_id\']=[f\'train{i}\' for i in range(n)];train[\'site_key\']=np.repeat(np.arange(10),6).astype(str)\\n    train[TARGETS]=rng.normal(size=(n,5));test[\'sample_id\']=[f\'test{i}\' for i in range(4)]\\n    sample=test[[\'sample_id\']].copy();sample[TARGETS]=0.\\n    monkeypatch.setattr(r,\'load_tables\',lambda root:(train,test,sample,names))\\n    values={name:rng.normal(size=(n+4,w)) for name,w in dict(official=54,quality=4,brightness=6,thermal=12,scrambled=12).items()}\\n    monkeypatch.setattr(r,\'matrices\',lambda *args:(values,{}))\\n    monkeypatch.setattr(r,\'safe_fit\',lambda x,y,z:np.full(len(z),np.nanmean(y)))\\n    calls=[]\\n    def export(sample,p,path):calls.append(str(path));return \'wrong-frozen-identity\'\\n    monkeypatch.setattr(r,\'export_submission\',export)\\n    with pytest.raises(ValueError,match=\'Frozen final control CSV identity mismatch\'):r.run(tmp_path,tmp_path)\\n    assert len(calls)==1 and calls[0].endswith(\'private_control_replay.csv\')\\n    assert not (tmp_path/\'refit_receipt.json\').exists()\\n"}')
    expected=json.loads('{"spectral_lab/refit.py": "1e9607e06942e62248f493b587c93559306dcd77cb841fbb8a9f87af7fb6e86f", "test_refit.py": "566f8274c91c2e7f43e68582c76c0f1ac68dce8da3606f933536b42d4dec4c3b"}')
    for name,code in sources.items():
        if hashlib.sha256(code.encode()).hexdigest()!=expected[name]:raise ValueError('Refit source identity mismatch')
        path=workspace/name;path.parent.mkdir(parents=True,exist_ok=True);path.write_text(code)
    (output/'refit_source_freeze.json').write_text(json.dumps(expected,sort_keys=True))
    check=subprocess.run([sys.executable,'-m','pytest',str(workspace),'-q'],capture_output=True,text=True)
    print(check.stdout[-2000:]);print(check.stderr[-1000:])
    if check.returncode:raise RuntimeError('Accepted-refit regression check failed')
    from spectral_lab.refit import run as accepted_refit
    refit_receipt=accepted_refit(root,output)


    # Run only as the final cell of the fresh saved private notebook.
    import json,hashlib
    import numpy as np
    import pandas as pd
    expected='ae6287bd5aee9269d6d5eb6ffe8e77f7b60db3714c7b275e59154474a2a2055a'
    receipt=json.loads((output/'spectral_receipt.json').read_text())
    refit=json.loads((output/'refit_receipt.json').read_text())
    audit=json.loads((output/'independent_spectral_audit.json').read_text())
    if not receipt['gate']['passed'] or not audit['passed']:raise ValueError('Replayed gate/audit failed')
    if abs(receipt['development']['scores']['F']['mcrmse']-.9194996153136487)>1e-12:raise ValueError('Development replay mismatch')
    if abs(receipt['environment_stress']['scores']['F']['mcrmse']-.9462006522193394)>1e-12:raise ValueError('Environment replay mismatch')
    if refit['candidate_sha256']!=expected or hashlib.sha256((output/'submission_spectral.csv').read_bytes()).hexdigest()!=expected:
        raise ValueError('Final candidate identity mismatch')
    csv=pd.read_csv(output/'submission_spectral.csv',dtype={'sample_id':str})
    template=pd.read_csv(root/'sample_submission.csv',dtype={'sample_id':str})
    if list(csv.columns)!=list(template.columns) or not csv.sample_id.equals(template.sample_id) or len(csv)!=1221 or not np.isfinite(csv.iloc[:,1:].to_numpy()).all():
        raise ValueError('Final CSV independent contract mismatch')
    old=pd.read_csv(output/'private_control_replay.csv',dtype={'sample_id':str})
    if not np.array_equal(csv.iloc[:,1:5].to_numpy(),old.iloc[:,1:5].to_numpy()):raise ValueError('Protected exported targets changed')
    replay={'schema':1,'expected_results_matched':True,'candidate_sha256':expected,'rows':len(csv),
            'protected_exported_targets_equal':True,'development_mcrmse':receipt['development']['scores']['F']['mcrmse'],
            'environment_mcrmse':receipt['environment_stress']['scores']['F']['mcrmse'],
            'note':'Fresh process established by the saved Save and Run All version, not by this in-process assertion alone.',
            'no_scored_submission_in_notebook':True}
    (output/'replay_verification.json').write_text(json.dumps(replay,sort_keys=True,indent=2))
    print('SPECTRAL_REPLAY_BEGIN');print(json.dumps(replay,sort_keys=True));print('SPECTRAL_REPLAY_END',flush=True)


## Accepted spectral result and reproduction

The single frozen spectral submission completed with **public MCRMSE 1.13057**, improving the old HistGB result of 1.15808 by 2.38% on the public subset. Private performance remains unknown. The new spectral candidate and original HistGB are selected as the two finalists; CatBoost is unselected. No post-score parameter or ensemble tuning was performed.

Fresh private Save and Run All version **373122625** succeeded in **767.2 seconds**, reproducing development/stress scores and the exact 1,221-row CSV hash. Nine original checks and two refit checks passed. The CSV and all competition observations remain private; this source notebook has no output files.

The [complete source, tests, source freezes, aggregate receipts, protocol and negative-control discussion](https://github.com/Yoriichi-GOD/LavaLabs/tree/main/research/obscura/spectral-v4) are also available through the public LavaLabs repository. This link shares the supporting repository code through Kaggle's public channel. The [current research report and evidence interface](https://yoriichi-god.github.io/LavaLabs/#evidence) summarize the same experiment.

The shuffled control nearly matches the full descriptors, so a causal spatial-structure claim is unsupported. These engineering features are not calibrated fire detections, physical emissions or an independently validated future-time model. SI-assisted code, testing, analysis and writing; organizer observations are unchanged.
